# Neuropixels preprocessing: crop → spikes (Kilosort4) + curation + LFP

This notebook takes one Open Ephys Neuropixels recording, crops off the start, and prepares two independent outputs: a spike-band recording sorted with Kilosort4 (with optional curation), and an LFP recording.

It can also load analog (ADC) streams, a OneBox ADC, an NI-DAQ or both (`ADC_MODE`), for inspection and, when both streams demonstrably share a clock, crop it to the same time interval (cell 1.2, off by default). It contains **no** opto-artifact removal, FaceMap/behavior processing, or analysis of the ADC signals.

**Strict inputs:** missing Open Ephys files or missing outputs of an earlier step stop the cell that needs them, with the exact path and the step that makes it. Nothing is reconstructed, substituted or stamped after the fact (for example, `sample_numbers.npy` is never rebuilt from `timestamps.npy`).

### Run order

| Step | Cell | What it does |
|---|---|---|
| 0 | **0.1** Imports | Loads the analysis module with autoreload |
| 0 | **0.2** Config | All user settings, including `REMOVE_FIRST_MINUTES` and `EVENT_CHANNEL` |
| 1 | **1.1** Load + crop + shared preprocessing | Produces `rec_step4`, the input to both branches; saves events cropped to the recording |
| 1 | **1.2** ADC stream *(optional)* | Loads the analog stream with its own rate and units; crops it to the retained interval only if a shared clock is verified |
| 2 | **S1** Spike-band filter | 300 Hz high-pass + global median CMR |
| 2 | **S2** Save spike-band | Writes the recording Kilosort reads, with its settings file |
| 2 | **S3** Trim timestamps | Reads `timestamps.npy` **and** `sample_numbers.npy`, writes cropped copies, checks for gaps |
| 2 | **S4** Kilosort4 | Needs S3; checks Docker, runs the sorter, saves a run record |
| 2 | **S5** Register Phy curation *(optional)* | Only after merging/splitting in Phy; then rerun C1–C5 |
| 3 | **C1** Sorting analyzer *(optional)* | Waveforms, templates, quality and template metrics |
| 3 | **C2** UnitRefine *(optional)* | Noise / SUA / MUA labels with real Kilosort IDs |
| 3 | **C3** 16-bit file *(optional)* | `.dat` copy of the spike band for MATLAB tools, with safety checks |
| 3 | **C4** MATLAB scripts *(optional)* | Writes `run_bombcell.m` and `run_cellexplorer.m`; run them in MATLAB |
| 3 | **C5** Unit summary *(optional)* | One row per cluster, merging every result that exists |
| 4 | **L1** LFP filter | 1–300 Hz, resampled to 1250 Hz |
| 4 | **L2** Save LFP | Writes no-ref and CMR versions, with settings files |
| 5 | **A1** Spike–LFP alignment *(beta, untested)* | Puts spikes and LFP on the Open Ephys clock |

Always run **0.1 → 0.2 → 1.1** first (1.2 only if you load an ADC stream). After that, the Spike branch (S1–S4, in order) and LFP branch (L1–L2) are independent: run either one, or both, in any order. The curation cells (C1–C5) need S4. C1 → C2 go together, and C3 → C4 → MATLAB → C5 go together. Run A1 once S4 and L2 are done if you want to analyse spikes against the LFP.

### Files written

| File / folder | Location | Written by |
|---|---|---|
| `recording_crop_timing.npz` | `data_folder/custom_analytics/` | 1.1 |
| `adc/<key>/<folder>/adc_alignment_info.json` (one per ADC source) | `data_folder/custom_analytics/` | 1.2 |
| `shifted_events_block{N}.npy`, `enumerated_shifted_events_block{N}.txt` | `data_folder/custom_analytics/` | 1.1 |
| `cropped_events_block{N}.npy`, `_info.json`, `cropped_event_indices_block{N}.npy`, `cropped_event_in_gap_block{N}.npy` | `data_folder/custom_analytics/` | 1.1 |
| `si_preprocessed/` (+ `nam_preprocessing_settings.json`) | preprocessed folder | S2 |
| `timestamps_trimmed.npy` (+ `.settings.json`), `sample_numbers_trimmed.npy`, `timestamp_gap_report.json` | preprocessed folder | S3 |
| Kilosort output (`sorter_output/`), `nam_kilosort_run_record.json` | kilosort folder | S4 |
| `nam_phy_curation_record.json` | kilosort folder | S5 |
| `si_analyzer_<sort id>/` | data folder | C1 |
| `unit_refine/unit_refine_results.csv` (+ `.settings.json`) | data folder | C2 |
| `matlab_curation/spikeband_int16.dat` (+ `.settings.json`) | data folder | C3 |
| `matlab_curation/run_bombcell.m`, `run_cellexplorer.m` | data folder | C4 |
| `bombcell/bombcell_<id>/cluster_bc_unitType.tsv` (+ `.settings.json`) | data folder | Bombcell, in MATLAB |
| `matlab_curation/cellexplorer_<id>/cellexplorer_<id>.cell_metrics.cellinfo.mat` (+ `.settings.json`) | data folder | CellExplorer, in MATLAB |
| `unit_summary.csv`, `consensus_good_units.csv` | `data_folder/custom_analytics/` | C5 |
| `si_preprocessed_lfp_noref/`, `si_preprocessed_lfp_cmr/` (+ settings files) | preprocessed folder | L2 |
| `alignment/` (LFP timestamps, spike times in s, spike→LFP index) | preprocessed folder | A1 |

### Safety checks built in
- **Settings files:** S2 and L2 save the crop, bad-channel and filter settings with each folder. An existing folder is reused only if its length **and** settings match; otherwise the cell stops and names each setting that changed. A folder without a settings file (older notebook version or interrupted save) always stops the cell: move or delete it and rerun. Settings are only ever written when a cell produces a new output.
- **Event channel and bad channels:** 1.1 stops with the real channel names if `EVENT_CHANNEL` or any `bad_channels_list` entry isn't found.
- **Timestamps:** S3 needs the recording's own `timestamps.npy` and `sample_numbers.npy` and stops if either is missing (Open Ephys < 0.6 recordings, which have no `sample_numbers.npy`, can't be used). It reports missing samples, repeats, backward jumps and NaN timestamps in the cropped range and records where the files came from.
- **Kilosort:** S4 refuses to overwrite an existing output, requires S3's trimmed timestamps (read, not inferred, same crop and channels as S2, same length), and records the Docker image and parameters.
- **Curation:** C1 stops if its analyzer was built from a different Kilosort run. C3 checks disk space, 16-bit range, channel order and read-back. Every curation result is tagged with a fingerprint of the Kilosort output; C2, the MATLAB scripts and C5 stop if it doesn't match the current sort.
- **Kilosort input:** C1, C3, C5 and A1 check S4's run record so the S2 folder in use is the exact one Kilosort sorted.
- **Alignment:** A1 refuses S3 outputs with inferred sample numbers or no provenance, stops on repeated or backward samples, masks the LFP around forward gaps, and checks that spikes, LFP, timestamps and the Kilosort run share one crop and channel set.
- **ADC stream (1.2):** uses the stream's own sampling rate and stated units (raw values when `structure.oebin` gives no unit), never the probe's 0.195 µV/bit. It is cropped only when the same TTL pulses carry the same timestamps in both streams and the ADC covers the whole retained interval; otherwise it can be inspected but `rec_adc` is `None`.

---
## 0 · Setup

### 0.1 Imports
Loads `Neuropixel_analysis_module` and turns on autoreload, so edits to the module are picked up without restarting the kernel.

**Edit:** nothing.

In [ ]:
# ==========================================================================
# [0.1] Imports
# ==========================================================================

%load_ext autoreload
%autoreload 2

import Neuropixel_analysis_module as nam

### 0.2 Configuration
Every setting you normally change lives here. It checks that the recording folders exist (and that every ADC source selected by `ADC_MODE` can be found) **before** writing `nam_analysis_config.json`, which every later cell reads through `config_path`.

**Edit:**
- `REMOVE_FIRST_MINUTES`: minutes cut from the start (applies to spikes **and** LFP).
- `base_path`, `record_node`, `experiment_label`, `recording_label`, `Neuropix_PXI_label`: which recording and probe stream to load.
- `ADC_MODE` and `ADC_SOURCES`: the optional analog streams loaded in 1.2. `ADC_MODE` is exactly one of `'off'`, `'nidaq'`, `'onebox'`, `'both'` (anything else stops 0.2). `ADC_SOURCES` holds, per device, the folder under `continuous` (`label`, e.g. `'OneBox-121.OneBox-ADC'` or the session's NI-DAQ folder such as `'NI-DAQmx-104.PCIe-6251'`), optional `channel_ids` (IDs as 1.2 lists them), `stream_name` (only if 0.2 reports two reader streams in one folder) and `unit_if_not_stated` (`'V'` only if you know it). Only the selected sources are checked: a missing NI-DAQ doesn't matter for `'onebox'`, but `'both'` stops if either is missing, naming the expected folder and the folders and streams found. 0.2 finds each source's reader stream name (for a OneBox: `'<Record Node>#OneBox-121.OneBox-ADC_ADC'`). Both sources must be in the probe's Record Node; another Record Node is refused. (`LOAD_ADC` / `ADC_label` and `NIDAQ_label` from older versions are replaced; older configuration files still load: single-ADC ones as one source, NI-DAQ-only ones as `'off'`.)
- `bad_channels_list`: probe channels to drop, e.g. `['CH96', 'CH124']`.
- `EVENT_CHANNEL`: which Open Ephys event channel 1.1 reads (independent of `ADC_MODE`). The default, `"Neuropixels PXI Sync"`, is the probe's 1 Hz sync signal, **not** a stimulus. If the name isn't found, 1.1 stops and lists the channels that exist.
- `EVENT_LINE`: one event channel can contain several TTL lines (e.g. an NI-DAQ digital input). `None` keeps all lines; a label such as `"1"` keeps only that line. 1.1 prints the labels in the channel and warns if several are combined.

**Makes:** `config_path`, `REMOVE_FIRST_MINUTES`, `bad_channels_list`, `EVENT_CHANNEL`, `EVENT_LINE`.

In [ ]:
# ==========================================================================
# [0.2] Configuration — run this before anything else
# Makes: config_path, REMOVE_FIRST_MINUTES, bad_channels_list, EVENT_CHANNEL, EVENT_LINE
# ==========================================================================

import Neuropixel_analysis_module as nam
from pathlib import Path
# === USER INPUT SECTION ===
# Set your parameters clearly in this section:

# Minutes to skip at the start of the recording (applies to spikes AND LFP).
# Must be shorter than the recording (1.1 stops otherwise).
REMOVE_FIRST_MINUTES = 0

# uV per bit, used ONLY if the recording carries no gains (Open Ephys recordings do: 'bit_volts' in
# structure.oebin, 0.195 for the Neuropixels probe channels). C3 warns if this differs from the recording.
# Probe only: never applied to the ADC stream.
bit_to_uv_probe_IMEC_calibration_value = 0.195

# EDIT: the Open Ephys session folder (the one that contains 'Record Node ...').
# Keep the path short: Windows stops at 260 characters and outputs go ~180 deeper.
base_path = Path.cwd() / 'test_data' / '2026-0822_473nm_optotest_cosine' / '2026-08-22_02-27-46'
# EDIT: copy these names exactly from the folders inside base_path
record_node = 'Record Node 112'                 # base_path\<record_node>
experiment_label = 'experiment1'                # ...\<record_node>\<experiment_label>
recording_label = 'recording1'                  # ...\<experiment_label>\<recording_label>
Neuropix_PXI_label = 'Neuropix-PXI-104.ProbeA'  # ...\<recording_label>\continuous\<probe folder>
# (probe folder: e.g. 'Neuropix-PXI-100.ProbeA' with a PXI basestation, 'OneBox-121.ProbeA' with a OneBox)

# Analog (ADC) streams loaded in cell 1.2, kept apart from the probe processing.
# ADC_MODE: exactly one of 'off', 'nidaq', 'onebox', 'both'. Only the selected sources are checked;
# a selected source that doesn't exist stops 0.2. 'both' loads each source separately and checks
# each one's clock against the probe on its own (they are never assumed to share a clock).
ADC_MODE = 'off'
ADC_SOURCES = {
    'onebox': {
        'label': 'OneBox-121.OneBox-ADC',    # folder in ...\continuous\ (check it on disk)
        'channel_ids': None,                 # None = all; or IDs as 1.2 lists them, e.g. ['ADC0', 'ADC3']
        'stream_name': None,                 # leave None unless 0.2 reports two reader streams in the folder
        'unit_if_not_stated': None,          # 'V' only if you KNOW the unit and structure.oebin gives none
    },
    'nidaq': {
        'label': 'NI-DAQmx-104.PCIe-6251',   # example only: copy the real folder name
        'channel_ids': None,                 # e.g. ['AI0', 'AI3']
        'stream_name': None,
        'unit_if_not_stated': None,
    },
}  # end ADC_SOURCES

configuration_file = 'nam_analysis_config.json'
#bad_channels_list = ['CH96', 'CH124', 'CH303', 'CH310', 'CH348']
bad_channels_list = []
block_index_val = 0
exp_rec = "exp1_rec1"     # label only, not used by this notebook
probe_bank = "p0_bA"

# Open Ephys event channel read in 1.1. The default is the probe's 1 Hz sync
# signal, NOT a stimulus. If the name is wrong, 1.1 stops and lists the real names.
EVENT_CHANNEL = "Neuropixels PXI Sync"
# One Open Ephys event channel can hold several TTL lines (e.g. an NI-DAQ digital input).
# None = keep every line; or the line label to keep, e.g. "1". 1.1 prints the labels found.
EVENT_LINE = None

# === END USER INPUT SECTION ===

# Check the recording folders BEFORE anything is written
recording_root = base_path / record_node / experiment_label / recording_label
_session_check = recording_root / "continuous" / Neuropix_PXI_label
if not _session_check.is_dir():
    raise FileNotFoundError(
        "The recording folder set above doesn't exist:\n"
        f"{_session_check}\n"
        "Check base_path, record_node, experiment_label, recording_label and Neuropix_PXI_label "
        "against the folder names on disk."
    )

# Stops with the expected path and the folders / streams found if a selected source doesn't match
adc_sources_resolved = nam.resolve_adc_sources(
    recording_root, experiment_label, record_node, ADC_MODE, ADC_SOURCES, probe_label=Neuropix_PXI_label,
)
if not adc_sources_resolved:
    print("ADC streams: none (ADC_MODE = 'off')")
for _src in adc_sources_resolved:
    print(f"ADC source '{_src['key']}': folder {_src['label']} -> reader stream {_src['stream_name']!r} "
          f"(id {_src['stream_id']})")

# Create the configuration file
nam.save_nam_analysis_config(
    base_path=base_path,
    record_node=record_node,
    experiment_label=experiment_label,
    recording_label=recording_label,
    Neuropix_PXI_label=Neuropix_PXI_label,
    bit_to_uv_probe_IMEC_calibration_value=bit_to_uv_probe_IMEC_calibration_value,
    bad_channels_list=bad_channels_list,
    block_index_val=block_index_val,
    overwrite=True,
    adc_mode=ADC_MODE,
    adc_sources=adc_sources_resolved,
    configuration_file=configuration_file,
    remove_first_minutes=REMOVE_FIRST_MINUTES,
)

config_path = nam.get_config_path(
    base_path,
    record_node,
    experiment_label,
    recording_label,
    configuration_file=configuration_file,
)

print("Config path:", config_path)
print("Skipping first", REMOVE_FIRST_MINUTES, "min")
print("Event channel:", EVENT_CHANNEL, "(independent of ADC_MODE)")
print("ADC mode:", ADC_MODE)

---
## 1 · Shared preprocessing
Everything here is used by **both** the spike and LFP branches.

### 1.1 Load, crop, and shared preprocessing
1. Loads the full Neuropixels stream from Open Ephys (PXI basestation or OneBox probe folder, e.g. `Neuropix-PXI-100.ProbeA` or `OneBox-121.ProbeA`).
2. Cuts off the first `REMOVE_FIRST_MINUTES` by frame, and saves the exact crop start to `recording_crop_timing.npz` (S3 uses it to trim timestamps). This is the probe's crop only; an ADC stream is cropped in 1.2 from its own timestamps.
3. Runs `nam.process_block` (block time shift and event extraction) on the `EVENT_CHANNEL` set in 0.2. It prints the TTL lines in that channel (label: number of rising edges); `EVENT_LINE` keeps one line, otherwise all lines are combined with a warning. Event durations (rising to falling edge) are saved in `cropped_event_durations_block{N}.npy`.
4. Removes bad channels. Any ID in `bad_channels_list` that isn't in the recording stops the cell (e.g. `96` instead of `'CH96'`), so an intended exclusion can't be skipped silently.
5. Applies phase shift (corrects the Neuropixels ADC sampling offset between channels).
6. Saves the shifted event times as `.npy` and as a readable `.txt` list (rewritten every run so it always matches the current crop). These include events from the removed minutes, with negative times.
   Event times are placed on the probe's **sample clock**: each event is located in the probe's own `timestamps.npy`, then shifted by the crop, `(sample position − crop start frame) / fs`. That's the clock of Kilosort's `spike_times / fs`, even across dropped samples. (Subtracting the recording's start time, as before, can mix clocks: Neo takes the start from `sample_numbers`, while Open Ephys ≥ 0.6 event times use the synchronized clock.) The cell prints how far the two methods differ. Events before the probe's first or after its last sample get no position (NaN); nothing is extrapolated. Dropped-sample gaps are taken from the probe's `sample_numbers.npy` when present. If the probe isn't the main sync stream (e.g. a OneBox session), its timestamps contain small Open Ephys synchronization corrections; backward steps below `nam.SYNC_CORRECTION_MAX_S` (0.25 ms) are accepted and counted (info file), larger ones stop the cell. Only the **retained** part of the timestamps must be valid: if timestamps repeat or run backwards only in the discarded first minutes, the cell maps events with the retained part alone and sets events from the discarded part to NaN (counted in the info file).
7. Saves a cropped copy, `cropped_events_block{N}.npy`: only events inside the cropped recording, timed from the same zero as Kilosort's spike times. `cropped_events_block{N}_info.json` records how many events fell in the removed part, and `cropped_event_indices_block{N}.npy` maps each kept event back to its position in the full list, and `cropped_event_in_gap_block{N}.npy` flags events that fall inside a dropped-sample gap (gaps come only from the probe's `sample_numbers.npy`; without it they are reported as not determined and no gap file is written). `recording_crop_timing.npz` is written only after the event channel, timestamps and bad channels have been checked.

**Edit:** nothing (settings come from 0.2).

**Makes:** `rec_step4` (the shared input to S1 and L1), `shared_settings`, `cropped_events_block`.

In [ ]:
# ==========================================================================
# [1.1] Load, crop, block shift, bad channels, phase shift
# Needs: 0.2
# Makes: rec_step4, shared_settings, PROBE_CROP_START_FRAME, PROBE_CROP_START_S,
#        cropped_events_block
# ==========================================================================

from pathlib import Path
import numpy as np
import spikeinterface.full as si
import Neuropixel_analysis_module as nam

# REMOVE_FIRST_MINUTES and bad_channels_list come from cell 0.2
if "REMOVE_FIRST_MINUTES" not in globals():
    raise NameError("Run cell 0.2 first (REMOVE_FIRST_MINUTES is defined there).")

# ============================================================
# LOAD CONFIG
# ============================================================

config = nam.load_experiment_config(config_path)

recording_root = (
    Path(config["base_path"])
    / config["record_node"]
    / config["experiment_label"]
    / config["recording_label"]
)

custom_analytics_dir = Path(config["data_folder"]) / "custom_analytics"

# ============================================================
# LOAD FULL OPEN EPHYS NEUROPIXELS RECORDING
# ============================================================

rec_step1_full = si.read_openephys(
    folder_path=recording_root,
    experiment_names=config["experiment_names_str"],
    stream_name=config["neuropix_stream_name_str"],
    load_sync_timestamps=False,
    load_sync_channel=False,
    # False: with Neo 0.14, per-channel annotations fail to load when the recording also has
    # an ADC-only stream (e.g. a OneBox 'OneBox-ADC' stream); nothing below uses them
    all_annotations=False,
)

print("\n=== Original Full Recording ===")
print(rec_step1_full)

original_duration_min = rec_step1_full.get_total_duration() / 60
print(f"\nOriginal duration: {original_duration_min:.2f} min")

# ============================================================
# CROP FIRST REMOVE_FIRST_MINUTES MINUTES
# ============================================================

fs = float(rec_step1_full.get_sampling_frequency())
total_samples = rec_step1_full.get_num_samples()

start_frame = int(round(REMOVE_FIRST_MINUTES * 60 * fs))

if start_frame >= total_samples:
    raise ValueError(
        f"Recording is only {original_duration_min:.2f} min long. "
        f"Cannot remove {REMOVE_FIRST_MINUTES} min."
    )

# Preserve exact probe crop origin (used for timestamp trimming)
PROBE_CROP_START_FRAME = int(start_frame)
PROBE_CROP_START_S = PROBE_CROP_START_FRAME / fs
# (The ADC stream is cropped in 1.2 from its OWN timestamps; nothing about it is assumed here.)

rec_step1 = rec_step1_full.frame_slice(
    start_frame=start_frame,
    end_frame=None,
)

cropped_duration_min = rec_step1.get_total_duration() / 60
removed_samples = rec_step1_full.get_num_samples() - rec_step1.get_num_samples()
removed_minutes = removed_samples / fs / 60

print("\n=== Cropped Recording Used For All Downstream Processing ===")
print(rec_step1)
print(f"\nDuration after cropping: {cropped_duration_min:.2f} min")
print(f"Removed samples: {removed_samples:,}")
print(f"Removed duration: {removed_minutes:.2f} min")
print(f"Probe crop start frame: {PROBE_CROP_START_FRAME:,}")
print(f"Probe crop start time: {PROBE_CROP_START_S:.6f} s")


# Quick look at the cropped recording (comment out to skip the plot)
nam.inspect_and_plot_recording(rec_step1, si)

# ============================================================
# PROCESS BLOCK ON CROPPED RECORDING
# ============================================================

if "EVENT_CHANNEL" not in globals():
    raise NameError("Run cell 0.2 first (EVENT_CHANNEL is defined there).")

rec_step2, events_block, shifted_events_block = nam.process_block(
    base_path=recording_root,
    block_index=config["block_index_val"],
    rec=rec_step1,
    event_channel=EVENT_CHANNEL,
)

# ============================================================
# EVENT TIMES ON THE PROBE'S SAMPLE CLOCK
# process_block subtracts the recording's start time, which Neo takes from
# sample_numbers[0] / fs. Open Ephys (>= 0.6) event times are on the synchronized
# clock (timestamps.npy), so the two origins can differ, and dropped samples break
# "time - start" further. Instead, each event is placed at its position in the
# probe's own timestamps, then shifted by the crop:
#     shifted time = (probe sample position - crop start frame) / fs
# This is the same clock as Kilosort's spike_times / fs, even across gaps.
# ============================================================

neuropix_folder_name = config.get("Neuropix_PXI_label", "Neuropix-PXI-100.ProbeA").split("#")[-1]
probe_timestamps_path = recording_root / "continuous" / neuropix_folder_name / "timestamps.npy"
if not probe_timestamps_path.exists():
    raise FileNotFoundError(f"Missing probe timestamps: {probe_timestamps_path}")
probe_timestamps_full = np.load(probe_timestamps_path, mmap_mode="r")
if len(probe_timestamps_full) != rec_step1_full.get_num_samples():
    raise ValueError(
        f"Probe timestamps.npy has {len(probe_timestamps_full):,} entries, the recording "
        f"{rec_step1_full.get_num_samples():,} samples."
    )

# Gaps (dropped samples) come from the probe's sample_numbers.npy when it exists; otherwise from
# timestamp steps (recorded in the info file as gap_source)
probe_sample_numbers_path = probe_timestamps_path.with_name("sample_numbers.npy")
probe_sample_numbers_full = np.load(probe_sample_numbers_path, mmap_mode="r") if probe_sample_numbers_path.exists() else None
event_clock_report = {}

# One event channel can hold several TTL lines; Neo keeps the line in the event label.
_events = events_block.get_events(channel_id=EVENT_CHANNEL)
_line_labels, _line_counts = np.unique(_events["label"], return_counts=True)
print(f"TTL lines in '{EVENT_CHANNEL}' (label: rising edges):",
      {str(l): int(n) for l, n in zip(_line_labels, _line_counts)})
_event_line = globals().get("EVENT_LINE", None)
if _event_line is None:
    _line_mask = np.ones(len(_events), dtype=bool)
    if _line_labels.size > 1:
        print(f"[warning] '{EVENT_CHANNEL}' has {_line_labels.size} TTL lines and their events are COMBINED. "
              "Set EVENT_LINE in 0.2 to keep one line.")
else:
    _line_mask = _events["label"] == str(_event_line)
    if not _line_mask.any():
        raise ValueError(
            f"EVENT_LINE {_event_line!r} not found in '{EVENT_CHANNEL}'. "
            f"Labels present: {[str(l) for l in _line_labels]}. Fix EVENT_LINE in 0.2."
        )
raw_event_times = np.asarray(_events["time"][_line_mask], dtype=float)
event_durations_s = np.asarray(_events["duration"][_line_mask], dtype=float)
event_labels = np.asarray(_events["label"][_line_mask]).astype(str)
shifted_events_block = np.asarray(shifted_events_block, dtype=float)[_line_mask]   # same order as get_events
n_events_unmappable = 0
# Events before the probe's first or after its last sample get no position (NaN): nothing is extrapolated
try:
    event_sample_full, event_in_gap, event_before, event_after = nam.events_to_probe_samples(
        raw_event_times, probe_timestamps_full, fs, extrapolate=False, return_sides=True,
        sample_numbers=probe_sample_numbers_full, report=event_clock_report,
    )
except ValueError as exc_full:
    # Invalid timestamps may sit only in the discarded first minutes: then only the
    # retained part matters. Map with it, and leave earlier events unmapped (NaN).
    try:
        event_sample_full, event_in_gap, event_before, event_after = nam.events_to_probe_samples(
            raw_event_times, probe_timestamps_full, fs, first_sample=PROBE_CROP_START_FRAME,
            extrapolate=False, return_sides=True,
            sample_numbers=probe_sample_numbers_full, report=event_clock_report,
        )
    except ValueError as exc_kept:
        raise ValueError(
            "Event times can't be placed on the probe's clock: the probe's timestamps.npy has "
            f"repeated, backward or NaN values inside the RETAINED part of the recording.\n{exc_kept}\n"
            "Inspect with nam.check_timestamp_gaps(np.load(timestamps), np.load(sample_numbers), fs) "
            "before using this recording's event timing."
        ) from None
    n_events_unmappable = int(np.count_nonzero(np.isnan(event_sample_full) & ~event_after))
    print(
        "[warning] The probe's timestamps are invalid in the discarded first "
        f"{REMOVE_FIRST_MINUTES} min ({exc_full}). The retained part is valid and is used alone; "
        f"{n_events_unmappable} event(s) from the discarded part can't be placed and are set to NaN."
    )
event_outside = event_before | event_after
if event_clock_report.get("n_backward_corrections"):
    print(f"Probe timestamps: {event_clock_report['n_backward_corrections']} Open Ephys synchronization correction(s) "
          f"(backward steps up to {1e6 * event_clock_report['largest_backward_correction_s']:.1f} us; the probe is not "
          "the main sync stream). Accepted; events within a correction can be off by up to its size.")
old_shifted = np.asarray(shifted_events_block, dtype=float)
shifted_events_block = (event_sample_full - PROBE_CROP_START_FRAME) / fs

_both = np.isfinite(shifted_events_block) & np.isfinite(old_shifted)
clock_diff_ms = 1000 * np.max(np.abs(shifted_events_block - old_shifted)[_both]) if _both.any() else 0.0
print(f"Event clock check: start-time method vs probe-timestamp method differ by up to {clock_diff_ms:.3f} ms")
if clock_diff_ms > 1000 / fs:
    print("[warning] The recording's start time and the event clock disagree (or samples were dropped). "
          "Using the probe-timestamp mapping, which matches Kilosort's sample clock.")
if event_outside.any():
    print(f"[warning] {int(event_outside.sum())} event(s) fall before the probe's first or after its last "
          "sample: no position (NaN), not extrapolated.")

# ============================================================
# REMOVE BAD CHANNELS
# ============================================================

available_channels = rec_step2.get_channel_ids()

try:
    bad_channels_list
except NameError:
    bad_channels_list = []

unmatched_bad_channels = [ch for ch in bad_channels_list if ch not in available_channels]
if unmatched_bad_channels:
    raise ValueError(
        f"bad_channels_list has IDs that are not in this recording: {unmatched_bad_channels}\n"
        f"Channel IDs look like: {list(available_channels[:5])} ... {list(available_channels[-2:])}\n"
        "Fix bad_channels_list in 0.2 (e.g. 'CH96', not 96) and rerun 0.2 and 1.1."
    )
valid_bad_channels = [ch for ch in bad_channels_list if ch in available_channels]

print("\nBad channels requested:", bad_channels_list)
print("Bad channels removed:", valid_bad_channels)

rec_step3 = rec_step2.remove_channels(valid_bad_channels)

# ============================================================
# PHASE SHIFT
# ============================================================

rec_step4 = si.phase_shift(rec_step3)

# Settings shared by both branches (keys: nam.SHARED_SETTINGS_KEYS); S2 and L2 store them with each saved folder
shared_settings = {
    "recording_root": str(recording_root),
    "stream_name": config["neuropix_stream_name_str"],
    "remove_first_minutes": float(REMOVE_FIRST_MINUTES),
    "probe_crop_start_frame": int(PROBE_CROP_START_FRAME),
    "bad_channels_removed": [str(ch) for ch in valid_bad_channels],
    "phase_shift": True,
    "n_channels": int(rec_step4.get_num_channels()),
    "raw_sampling_rate_hz": float(rec_step4.get_sampling_frequency()),
    "raw_n_frames": int(rec_step4.get_num_frames()),
}

# ============================================================
# SAVE CROP METADATA (written only now, after the event channel, the probe's
# timestamps and the bad channels have been checked)
# ============================================================

# Save crop timing metadata (probe only)
custom_analytics_dir.mkdir(parents=True, exist_ok=True)
crop_timing_path = custom_analytics_dir / "recording_crop_timing.npz"

np.savez(
    crop_timing_path,
    remove_first_minutes=np.array(REMOVE_FIRST_MINUTES, dtype=float),
    probe_sampling_rate_hz=np.array(fs, dtype=float),
    probe_crop_start_frame=np.array(PROBE_CROP_START_FRAME, dtype=np.int64),
    probe_crop_start_s=np.array(PROBE_CROP_START_S, dtype=float),
    probe_n_frames_full=np.array(rec_step1_full.get_num_samples(), dtype=np.int64),
    recording_root=np.array(str(recording_root)),
)

print(f"Crop timing metadata saved to: {crop_timing_path}")

# ============================================================
# SAVE SHIFTED EVENTS
# ============================================================

sampling_rate = rec_step4.get_sampling_frequency()

block_index_val = config["block_index_val"]
shifted_events_str = f"shifted_events_block{block_index_val}.npy"

np.save(custom_analytics_dir / shifted_events_str, shifted_events_block)

output_text_file = custom_analytics_dir / f"enumerated_shifted_events_block{block_index_val}.txt"

# Always rewrite so the file matches the current crop setting
with open(output_text_file, "w") as f:
    for i, event in enumerate(shifted_events_block):
        f.write(f"{i}: {event - 0.7}, {event}, {event + 0.7}\n")
print(f"Enumerated shifted events saved to: {output_text_file}")

# ============================================================
# EVENTS INSIDE THE CROPPED RECORDING
# Events from the removed first minutes have negative times above. This copy keeps
# only events inside the cropped recording, timed from the same zero as
# Kilosort's spike_times / sampling_rate.
# ============================================================

import json

cropped_duration_s = rec_step4.get_total_duration()
cropped_events_block, cropped_event_indices = nam.crop_events_to_recording(
    shifted_events_block, cropped_duration_s
)
np.save(custom_analytics_dir / f"cropped_event_durations_block{block_index_val}.npy", event_durations_s[cropped_event_indices])
# Dropped-sample gaps only from the probe's own sample_numbers.npy. Without it they are NOT
# determined (no timestamp-based substitute): the gap file isn't written and the info says so.
gaps_known = probe_sample_numbers_full is not None
_gap_file = custom_analytics_dir / f"cropped_event_in_gap_block{block_index_val}.npy"
if gaps_known:
    cropped_event_in_gap = event_in_gap[cropped_event_indices]
    np.save(_gap_file, cropped_event_in_gap)
else:
    cropped_event_in_gap = np.zeros(cropped_event_indices.size, dtype=bool)
    print(f"[note] No sample_numbers.npy in the probe folder: dropped-sample gaps were not determined "
          f"(S3 also needs this file and will stop). {probe_sample_numbers_path}")
    if _gap_file.exists():   # this notebook's own output from an earlier run; it no longer applies
        _gap_file.unlink()
        print(f"[note] Removed {_gap_file.name} from an earlier run (it doesn't match this run).")
if gaps_known and cropped_event_in_gap.any():
    print(f"[warning] {int(cropped_event_in_gap.sum())} kept event(s) fall inside a dropped-sample gap; "
          f"see cropped_event_in_gap_block{block_index_val}.npy")

cropped_events_path = custom_analytics_dir / f"cropped_events_block{block_index_val}.npy"
np.save(cropped_events_path, cropped_events_block)

shifted_all = np.asarray(shifted_events_block, dtype=float).ravel()
cropped_events_info = {
    "event_channel": EVENT_CHANNEL,
    "event_line": None if _event_line is None else str(_event_line),
    "ttl_lines_in_channel": {str(l): int(n) for l, n in zip(_line_labels, _line_counts)},
    "durations_file": f"cropped_event_durations_block{block_index_val}.npy",
    "remove_first_minutes": float(REMOVE_FIRST_MINUTES),
    "cropped_duration_s": float(cropped_duration_s),
    "n_events_total": int(shifted_all.size),
    "n_events_in_removed_part": int(np.count_nonzero((shifted_all < 0) | (np.isnan(shifted_all) & ~event_after))),
    "n_events_after_recording_end": int(np.count_nonzero(shifted_all >= cropped_duration_s) + np.count_nonzero(event_after)),
    "n_events_kept": int(cropped_events_block.size),
    "kept_original_indices_file": f"cropped_event_indices_block{block_index_val}.npy",
    "n_events_in_dropped_sample_gaps": int(cropped_event_in_gap.sum()) if gaps_known else None,
    "n_events_outside_probe_recording": int(event_outside.sum()),
    "outside_events_position": "NaN (not extrapolated)",
    "gap_source": event_clock_report.get("gap_source") if gaps_known else "not determined (no sample_numbers.npy)",
    "probe_timestamp_sync_corrections": int(event_clock_report.get("n_backward_corrections", 0)),
    "largest_probe_timestamp_sync_correction_s": float(event_clock_report.get("largest_backward_correction_s", 0.0)),
    "n_events_unmappable_in_discarded_part": int(n_events_unmappable),
    "max_difference_vs_start_time_method_ms": float(clock_diff_ms),
    "time_zero": "first sample of the cropped recording (same as Kilosort spike_times / fs)",
    "mapping": "event time -> position in the probe's timestamps.npy -> (position - crop start frame) / fs",
}
np.save(custom_analytics_dir / cropped_events_info["kept_original_indices_file"], cropped_event_indices)
with open(custom_analytics_dir / f"cropped_events_block{block_index_val}_info.json", "w") as f:
    json.dump(cropped_events_info, f, indent=2)

print(
    f"Events: {cropped_events_info['n_events_total']} total, "
    f"{cropped_events_info['n_events_in_removed_part']} in the removed {REMOVE_FIRST_MINUTES} min, "
    f"{cropped_events_info['n_events_kept']} kept"
)
print(f"Cropped events saved to: {cropped_events_path}")

# ============================================================
# FINAL CHECKS
# ============================================================

print("\n=== Shared preprocessed recording (input to both branches) ===")
print(rec_step4)
print(f"rec_step1_full duration min:    {rec_step1_full.get_total_duration() / 60:.2f}")
print(f"rec_step4 cropped duration min: {rec_step4.get_total_duration() / 60:.2f}")
print(f"Full-recording frame corresponding to cropped frame 0: {PROBE_CROP_START_FRAME:,}")

### 1.2 Analog (ADC) streams (optional)
Handles every source selected by `ADC_MODE` in 0.2, one after the other and independently (after 1.1: the probe crop is read from this recording's `recording_crop_timing.npz`; a different crop left in memory stops the cell). With `ADC_MODE = 'off'` it does nothing and leaves the per-source dictionaries empty.

1. Finds the stream from 0.2 among the streams the installed reader discovers (folder name and reader stream name can differ) and checks that the Open Ephys files exist and agree: `continuous.dat`, `timestamps.npy`, `sample_numbers.npy` and the `structure.oebin` entry. Missing files stop the cell with their exact paths; nothing is reconstructed.
2. Loads each **whole** stream as `rec_adc_full_by_source[key]` with its **own** sampling rate and prints the stream, folder, rate, duration, channel IDs and units. Units come from `structure.oebin`; when it gives none (usual for Open Ephys), the channel keeps raw ADC values (`get_traces()` returns them by default; values scaled to µV are NaN). Set `ADC_SOURCES[...]['unit_if_not_stated'] = 'V'` in 0.2 only if you know the unit. The probe's 0.195 µV/bit is never applied. The ADC stream never enters phase shift, bad-channel removal, filtering, referencing or Kilosort, and no artifact removal is done.
3. Crops each source to the retained probe interval **only if its own shared clock with the probe is established** (two ADC sources are never assumed to share a clock just because both were loaded): both streams' timestamps are valid (Open Ephys writes −1 for samples it couldn't synchronize), the same TTL pulses (e.g. the sync line recorded by both devices) carry the same timestamps in both streams within `ADC_SYNC_TOLERANCE_MS`, and the ADC covers the whole retained interval. Equal rates or similar start times are not taken as evidence. Small backward timestamp steps (below `nam.SYNC_CORRECTION_MAX_S`, 0.25 ms) are allowed and reported: Open Ephys makes them in every stream that is synchronized to another (main) stream whenever it updates its mapping (the test recording's NI-DAQ stream has 9, at most 11 µs; in the 2026-09-06 OneBox session the OneBox ADC is the main stream and the probe's sync pulses deviate from it by up to 0.07 ms). Larger backward steps stop the alignment. The selected samples come from the ADC's own timestamps and indices; nothing is extrapolated. Repeated or backward ADC sample numbers stop the crop; forward gaps are reported (then use the timestamps, not frame ÷ fs).
4. If a source's clock can't be established, `rec_adc_by_source[key]` is `None` and the reason is printed; `rec_adc_full_by_source[key]` can still be inspected.

Times of cropped frames on the Kilosort / cropped-event clock: `nam.adc_cropped_times(adc_alignment_by_source[key], start, end)` (read from the ADC's `timestamps.npy`). `get_times()` of the recording is **not** that clock.

**Edit:** `ADC_SYNC_TOLERANCE_MS` (default 1 ms) at the top of the cell.

**Makes:** `rec_adc_full_by_source`, `rec_adc_by_source`, `adc_info_by_source`, `adc_alignment_by_source`, `adc_status_by_source` (keys `'onebox'` / `'nidaq'`; status `aligned`, `aligned_with_gaps` or `unavailable`) and one report per source, `custom_analytics/adc/<key>/<folder>/adc_alignment_info.json` (mode, source, files, selected bounds, evidence). With exactly one selected source, `rec_adc_full`, `rec_adc`, `adc_info`, `adc_alignment` and `ADC_ALIGNMENT_STATUS` are also set for it (as before); with `'both'` or `'off'` they are `None`.

In [ ]:
# ==========================================================================
# [1.2] Analog (ADC) streams: load, inspect, crop on the shared clock (optional)
# Needs: 0.2 (ADC_MODE, ADC_SOURCES), 1.1 (probe crop)
# Makes, per selected source key ('onebox', 'nidaq'):
#   rec_adc_full_by_source  whole stream (always inspectable)
#   rec_adc_by_source       samples covering the retained probe interval, or None if the
#                           shared clock with the probe isn't verified
#   adc_info_by_source, adc_alignment_by_source, adc_status_by_source
#   <data folder>/custom_analytics/adc/<key>/<folder>/adc_alignment_info.json
# Single-source aliases rec_adc_full, rec_adc, adc_info, adc_alignment, ADC_ALIGNMENT_STATUS
# are set only when exactly ONE source is selected; with 'both' (or 'off') they are None.
# ADC streams never enter the probe processing (phase shift, bad channels, filters,
# referencing, Kilosort) and no artifact removal is applied.
# ==========================================================================

import json
import numpy as np
from pathlib import Path
import Neuropixel_analysis_module as nam

# ----- settings -----
# Largest timestamp difference (ms) between the same TTL pulse recorded by the probe and an
# ADC stream for the two clocks to count as shared (checked for each source separately)
ADC_SYNC_TOLERANCE_MS = 1.0
# Longest time (s) allowed without a matched sync pulse: from the first retained probe sample to
# the first match, between matches, and from the last match to the end. 10 s suits a 1 Hz sync
# line; raise it only for a slower sync line. Longer -> alignment unavailable (not interpolated).
ADC_SYNC_MAX_GAP_S = 10.0
# (Units for channels without a stated unit: ADC_SOURCES[...]['unit_if_not_stated'] in 0.2.)
# --------------------

config = nam.load_experiment_config(config_path)
# Fresh every run: nothing from an earlier run or another ADC_MODE is reused
rec_adc_full_by_source, rec_adc_by_source = {}, {}
adc_info_by_source, adc_alignment_by_source, adc_status_by_source = {}, {}, {}
rec_adc_full = rec_adc = adc_info = adc_alignment = None
ADC_ALIGNMENT_STATUS = "disabled"

sources = config["adc_sources"]
print(f"ADC mode: {config['adc_mode']}  ({len(sources)} source(s): {[s_['key'] for s_ in sources]})")
if not sources:
    print("No ADC stream selected (ADC_MODE = 'off' in 0.2). The per-source dictionaries are empty.")
else:
    # Probe crop of THIS recording (its recording_crop_timing.npz from 1.1); a crop left in memory
    # from another recording or setting stops the cell instead of selecting the wrong ADC interval
    _probe_crop, _ = nam.resolve_probe_crop(config, globals())
    probe_ts_path = (Path(config["base_path"]) / config["record_node"] / config["experiment_label"]
                     / config["recording_label"] / "continuous" / config["Neuropix_PXI_label"] / "timestamps.npy")
    if not probe_ts_path.is_file():
        raise FileNotFoundError(f"Missing probe timestamps (written by Open Ephys): {probe_ts_path}")

    for src in sources:
        key = src["key"]
        # ---- load the whole stream (stops with exact paths / available names if anything is missing) ----
        rec_full, info = nam.load_adc_source(config, src)
        print(f"\n=== ADC source '{key}' ===")
        print(f"Folder:          {info['folder']}")
        print(f"Reader stream:   {info['stream_name']!r} (id {info['stream_id']})")
        print(f"Sampling rate:   {info['sampling_rate_hz']:g} Hz (its own; not the probe's)")
        print(f"Duration:        {info['duration_s']:.3f} s ({info['n_samples']:,} samples)")
        print(f"Channels found:  {info['channel_ids_found']}")
        print(f"Channels used:   {info['channel_ids_used']}")
        for u in info["units"]:
            print(f"  {u['channel_id']:>6}: bit_volts {u['bit_volts']:.6g}  unit: {u['physical_unit']}  ({u['status']})")
        if any(u["physical_unit"] == "not stated" for u in info["units"]):
            print("  -> channels without a stated unit: get_traces() gives raw ADC counts (the default); values "
                  f"scaled to uV are NaN until ADC_SOURCES['{key}']['unit_if_not_stated'] is set in 0.2.")
        if info["reader_has_probe"]:
            print("[note] The reader attached probe geometry to this ADC stream; it is not used here.")

        # ---- shared clock with the PROBE (this source alone) + crop ----
        al = nam.align_adc_to_probe(info, probe_ts_path, _probe_crop, info["recording_root"],
                                    config["Neuropix_PXI_label"], tolerance_ms=ADC_SYNC_TOLERANCE_MS,
                                    max_sync_gap_s=ADC_SYNC_MAX_GAP_S)
        rec_crop = None
        ev = al.get("sync_evidence", {})
        if al["status"] in ("aligned", "aligned_with_gaps"):
            rec_crop = rec_full.frame_slice(al["adc_start_index"], al["adc_end_index"])
            print(f"Shared clock with the probe inside the retained interval ({ev['probe_line']} vs {ev['adc_line']}): "
                  f"{ev['n_matched_in_window']}/{ev['n_adc_pulses_in_window']} ADC and "
                  f"{ev['n_probe_matched_in_window']}/{ev['n_probe_pulses_in_window']} probe TTL pulses match, "
                  f"longest unmatched run {ev['longest_unmatched_run']}, longest time without a match "
                  f"{max(ev['first_match_after_t0_s'], ev['last_match_before_t1_s'], ev['largest_interval_between_matches_s'] or 0.0):.1f} s, "
                  f"max difference {ev['max_abs_difference_ms']:.3f} ms "
                  f"({ev['n_matched']} matched in the whole session)")
            print(f"Cropped: ADC samples {al['adc_start_index']:,}-{al['adc_end_index']:,} "
                  f"({al['adc_n_selected']:,} samples); first one {al['adc_first_selected_minus_probe_t0_ms']:.3f} ms "
                  "after the first retained probe sample")
            _corr = al["adc_scan"]["n_small_backward_corrections"]
            if _corr:
                print(f"Open Ephys sync corrections in the ADC timestamps: {_corr} backward step(s), largest "
                      f"{1e6 * al['adc_scan']['largest_small_correction_s']:.1f} us "
                      f"(below {1e3 * nam.SYNC_CORRECTION_MAX_S:g} ms; allowed)")
            if al["status"] == "aligned_with_gaps":
                print("[warning]", al["reason"])
        else:
            print("[alignment unavailable]", al["reason"])
            print(f"rec_adc_full_by_source['{key}'] can be inspected; rec_adc_by_source['{key}'] = None.")

        rec_adc_full_by_source[key], rec_adc_by_source[key] = rec_full, rec_crop
        adc_info_by_source[key], adc_alignment_by_source[key], adc_status_by_source[key] = info, al, al["status"]

        # ---- one report per source, never shared ----
        out_dir = Path(config["data_folder"]) / "custom_analytics" / "adc" / key / src["label"]
        out_dir.mkdir(parents=True, exist_ok=True)
        rep_path = out_dir / "adc_alignment_info.json"
        with open(rep_path, "w") as f:
            json.dump(nam._jsonable({"adc_mode": config["adc_mode"], "source": src, "adc": info,
                                     "alignment": al}), f, indent=2)
        print("Report:", rep_path)

    if len(sources) > 1:
        print("\nEach source was checked against the probe separately; they are NOT assumed to share a clock "
              "with each other. Single-source aliases (rec_adc, adc_info, ...) are None: use the *_by_source "
              "dictionaries.")

    print("\n=== Summary ===")
    for key, st in adc_status_by_source.items():
        print(f"  {key:>7}: {st}" + ("" if rec_adc_by_source[key] is not None else
                                     f"  ({adc_alignment_by_source[key]['reason']})"))
    print("Times of cropped frames on the probe/Kilosort clock: nam.adc_cropped_times(adc_alignment_by_source[key], "
          "start, end) (rec.get_times() is NOT that clock).")

# Single-source compatibility aliases (only when exactly one source is selected)
if len(sources) == 1:
    _k = sources[0]["key"]
    rec_adc_full, rec_adc = rec_adc_full_by_source[_k], rec_adc_by_source[_k]
    adc_info, adc_alignment = adc_info_by_source[_k], adc_alignment_by_source[_k]
    ADC_ALIGNMENT_STATUS = adc_status_by_source[_k]
elif len(sources) > 1:
    ADC_ALIGNMENT_STATUS = None

---
## 2 · Spike branch → Kilosort4
Independent of the LFP branch. Requires 1.1.

### S1 Spike-band filter
High-pass at 300 Hz, then global median common reference (CMR). Because CMR is done here, Kilosort's own CAR is switched off in S4.

There is no low-pass by default: Kilosort's authors recommend high-pass only, because a 6000 Hz cut slightly reshapes spike waveforms. Set `SPIKE_FREQ_MAX = 6000.0` to get the old 300–6000 Hz bandpass back.

**Edit:** `SPIKE_FREQ_MIN`, `SPIKE_FREQ_MAX` (`None` = high-pass only).

**Makes:** `rec_step6_spike`, `spike_band_settings` (saved with the folder in S2).

In [ ]:
# ==========================================================================
# [S1] Spike-band filter (300 Hz high-pass) + global median CMR
# Needs: 1.1 (rec_step4)
# Makes: rec_step6_spike, spike_band_settings
# ==========================================================================

import spikeinterface.full as si

# ----- settings -----
SPIKE_FREQ_MIN = 300.0
SPIKE_FREQ_MAX = None   # None = high-pass only (recommended for Kilosort); 6000.0 = old bandpass
# --------------------

if SPIKE_FREQ_MAX is None:
    rec_step5_spike = si.highpass_filter(
        rec_step4,
        freq_min=SPIKE_FREQ_MIN,
        dtype="float32",
    )
    print(f"Filter: high-pass {SPIKE_FREQ_MIN:g} Hz")
else:
    rec_step5_spike = si.bandpass_filter(
        rec_step4,
        freq_min=SPIKE_FREQ_MIN,
        freq_max=SPIKE_FREQ_MAX,
        dtype="float32",
    )
    print(f"Filter: bandpass {SPIKE_FREQ_MIN:g}–{SPIKE_FREQ_MAX:g} Hz")

rec_step6_spike = si.common_reference(
    rec_step5_spike,
    operator="median",
    reference="global",
)

spike_band_settings = {
    **shared_settings,
    "spike_freq_min": SPIKE_FREQ_MIN,
    "spike_freq_max": SPIKE_FREQ_MAX,
    "common_reference": "global median",
    "dtype": "float32",
}

print("Spike-band recording:", rec_step6_spike)
print(f"Duration: {rec_step6_spike.get_total_duration() / 60:.2f} min")

### S2 Save spike-band recording
Writes the S1 output to disk; this is exactly what Kilosort reads in S4.

After saving, it writes `nam_preprocessing_settings.json` into the folder: the crop, bad channels, filter and reference settings used.

If the folder already exists, it is reused **only** if its length and its saved settings both match the current ones. If anything differs (e.g. a different crop or filter), the cell stops and names each setting that changed, e.g. `spike_freq_max: saved = None, current = 6000.0`.

A folder without a settings file (saved by an older notebook version, or a save that didn't finish) can't be checked, so the cell stops. Move or delete it and rerun; settings are never added to an existing folder after the fact.

**Edit:** nothing. The folder name comes from `processed_file_name` in the config (default `si_preprocessed`).

**Makes:** `si_save_path` on disk, with its settings file.

In [ ]:
# ==========================================================================
# [S2] Save spike-band recording for Kilosort
# Needs: S1 (rec_step6_spike)
# Makes: <preprocessed>/si_preprocessed/
# ==========================================================================

import shutil
import Neuropixel_analysis_module as nam
import spikeinterface.full as si

config = nam.load_experiment_config(config_path)
nam.define_analysis_paths_from_config(config_path)

output_folder = nam.preprocessed_folder

# S4 reads from this same folder name
processed_file_name = config.get("processed_file_name", "si_preprocessed")
si_save_path = output_folder / processed_file_name

if "spike_band_settings" not in globals():
    raise RuntimeError("Run S1 first (spike_band_settings missing).")
output_folder.mkdir(parents=True, exist_ok=True)

# Windows limits paths to 260 characters; SpikeInterface writes files ~40 characters
# below this folder, and later cells (Kilosort, CellExplorer) go deeper still
if len(str(si_save_path)) > 200:
    print(f"[warning] This path is {len(str(si_save_path))} characters long; Windows stops at 260 and "
          "later cells write deeper. If a save fails with 'filename or extension is too long', "
          "move the session folder to a short path (e.g. C:\\np\\<session>) and set base_path in 0.2.")

if not si_save_path.exists():
    try:
        rec_step6_spike.save(
            folder=si_save_path,
            chunk_duration="1s",
            n_jobs=-1,
            progress_bar=True,
        )
    except BaseException as exc:
        # The folder didn't exist before this save, so a failed save leaves only a broken copy: remove it
        shutil.rmtree(si_save_path, ignore_errors=True)
        still_there = si_save_path.exists()
        too_long = "too long" in str(exc).lower() or getattr(exc, "winerror", None) == 206
        raise RuntimeError(
            ("The path is too long for Windows (260-character limit).\n"
             f"Folder ({len(str(si_save_path))} characters): {si_save_path}\n"
             "Fix: move the session folder to a short path, e.g. C:\\np\\<session folder>,\n"
             "set base_path in 0.2 to it, then rerun 0.2 -> 1.1 -> S1 -> S2.\n"
             if too_long else
             f"Saving the spike-band recording failed: {type(exc).__name__}: {exc}\n")
            + ("The half-written folder was deleted." if not still_there else
               f"Delete this half-written folder by hand before rerunning:\n{si_save_path}")
        ) from exc
    # Written only after a complete save, so a half-written folder has no settings file
    _stamp = nam.write_settings_stamp(si_save_path, spike_band_settings)
    print(f"Spike-band recording saved to: {si_save_path}")
    print("Settings saved:", _stamp.name)
else:
    try:
        existing = si.load(si_save_path)
    except Exception as exc:
        raise RuntimeError(
            "An existing spike-band folder can't be loaded. It is probably half-written, "
            "left by an interrupted or failed save (e.g. a path-too-long error).\n"
            f"Delete this folder and rerun S2:\n{si_save_path}\n"
            f"(SpikeInterface said: {type(exc).__name__}: {exc})"
        ) from None
    if existing.get_num_frames() != rec_step6_spike.get_num_frames():
        raise ValueError(
            "An existing spike-band folder does not match the current crop "
            "(probably saved with a different REMOVE_FIRST_MINUTES).\n"
            f"saved frames   = {existing.get_num_frames():,}\n"
            f"current frames = {rec_step6_spike.get_num_frames():,}\n"
            f"Move or delete: {si_save_path}"
        )
    # Raises and lists every difference if the settings changed (e.g. the filter)
    status = nam.check_settings_stamp(si_save_path, spike_band_settings)
    if status == "no_stamp":
        # Never stamped after the fact: settings of an existing folder can't be known
        raise RuntimeError(
            "This folder has no settings file, so what it contains can't be checked:\n"
            f"{si_save_path}\n"
            "It was saved by an older notebook version, or the save didn't finish. Move or delete it, "
            "then rerun S2 to write a new one."
        )
    print(f"Folder already exists and matches current settings, skipping: {si_save_path}")

### S3 Trim timestamps and sample numbers
Kilosort outputs spike times as sample indices starting from 0 of the **cropped** recording. This cell cuts Open Ephys's original `timestamps.npy` and `sample_numbers.npy` to exactly the same range, so spike times can later be mapped back to real acquisition time.

**Both files are required and are read, never reconstructed.** If either is missing, the cell stops before writing anything, names the missing path, and says that the file comes from the Open Ephys recording. Recordings without `sample_numbers.npy` (Open Ephys < 0.6, or a copy that lost the file) can't be used by this workflow; restore the file from the original recording. If older S3 outputs whose sample numbers were *inferred* (earlier notebook versions) are present, the message says so; they are not deleted, and S4 and A1 refuse them.

It reads the crop start from 1.1 (or from `recording_crop_timing.npz` after a kernel restart) and takes the length and settings from the saved S2 folder, so it also works after a restart once S2 exists. Before S2 is saved, it uses S1's recording in memory instead. It checks the files' shapes and types, that both have the same length, and that the crop range lies inside them.

Before writing anything it checks the retained range: NaN/inf timestamps, repeated or backward sample numbers, or backward timestamp steps beyond Open Ephys's synchronization corrections stop the cell (S4 must not sort a recording whose clock is broken). Forward gaps (lost samples) are allowed and reported. The crop comes from this recording's `recording_crop_timing.npz`; a crop or preprocessing left in memory from another recording or setting stops the cell. It then writes the trimmed copies and `timestamp_gap_report.json` (`clock_ok_for_sorting: true`, which S4 requires), counting forward gaps (missing samples), repeated samples, backward jumps and NaN/inf timestamps separately, and saves `timestamps_trimmed.npy.settings.json`: the crop and channels plus the source files (`s3_provenance`, `sample_numbers: read`, and each source file's size, modification time and a sampled SHA-1), which S4 and A1 check: if either original file changes after S3, they refuse the outputs until S3 is rerun. It also stores SHA-256 checksums of its own three outputs, which S4 and A1 recompute, so an edited trimmed file or report is refused too. For a full checksum of the original files as well (slower: they are re-read completely), set `nam.STRICT_SOURCE_HASH = True` before S3. The outputs are written under temporary names and the settings file last, so an interrupted save leaves outputs that S4/A1 refuse. If the probe isn't the main sync stream (e.g. in a OneBox session where the OneBox ADC is the main stream), its timestamps contain Open Ephys synchronization corrections: small jumps where the sample numbers are continuous. They are counted separately (`n_sync_corrections`, `largest_sync_correction_s`) and don't count as gaps if smaller than `nam.SYNC_CORRECTION_MAX_S` (0.25 ms). A gap doesn't change Kilosort's sample indices, but it means sample ÷ fs is no longer exact acquisition time; the cell warns, and the trimmed timestamps (used by A1) stay correct.

**Edit:** nothing.

**Makes:** `timestamps_trimmed.npy`, `sample_numbers_trimmed.npy`, `timestamp_gap_report.json` in the preprocessed folder.

In [ ]:
# ==========================================================================
# [S3] Trim original timestamps / sample numbers to the cropped range
# Needs: Open Ephys timestamps.npy AND sample_numbers.npy in the probe folder;
#        S2 on disk (or S1 in memory); crop info from 1.1 or recording_crop_timing.npz.
#        Works after a kernel restart once S2 has been saved.
# Makes: timestamps_trimmed.npy (+ .settings.json), sample_numbers_trimmed.npy,
#        timestamp_gap_report.json
# Strict: both input files are READ; nothing is reconstructed or substituted.
# ==========================================================================

import json
import numpy as np
from pathlib import Path
import spikeinterface.full as si
import Neuropixel_analysis_module as nam

# ============================================================
# LOAD CONFIG / DEFINE PATHS
# ============================================================

config = nam.load_experiment_config(config_path)
nam.define_analysis_paths_from_config(config_path)

recording_root = (
    Path(config["base_path"])
    / config["record_node"]
    / config["experiment_label"]
    / config["recording_label"]
)
neuropix_folder_name = config["Neuropix_PXI_label"].split("#")[-1]
neuropix_continuous_dir = recording_root / "continuous" / neuropix_folder_name
original_timestamps_path = neuropix_continuous_dir / "timestamps.npy"
original_sample_numbers_path = neuropix_continuous_dir / "sample_numbers.npy"

timestamps_trimmed_path = nam.preprocessed_folder / "timestamps_trimmed.npy"
sample_numbers_trimmed_path = nam.preprocessed_folder / "sample_numbers_trimmed.npy"
gap_report_path = nam.preprocessed_folder / "timestamp_gap_report.json"

# ============================================================
# REQUIRED INPUTS (checked before anything is written)
# ============================================================

_missing = [p for p in (original_timestamps_path, original_sample_numbers_path) if not p.is_file()]
if _missing:
    _old = []
    if gap_report_path.exists():
        try:
            if json.loads(gap_report_path.read_text()).get("sample_numbers_inferred"):
                _old.append(
                    f"Existing S3 outputs in {nam.preprocessed_folder} were made by an older notebook version "
                    "that INFERRED sample numbers. They are not trusted (A1 refuses them) and were NOT deleted."
                )
        except Exception:
            pass
    raise FileNotFoundError(
        "Missing Open Ephys input file(s) for S3:\n  " + "\n  ".join(str(p) for p in _missing) + "\n"
        "Both timestamps.npy and sample_numbers.npy are written by the Open Ephys GUI (>= 0.6) during "
        "recording. This notebook reads them and does not reconstruct missing ones. Restore the file(s) "
        "from the original recording. (Recordings in the older format without sample_numbers.npy, Open "
        "Ephys < 0.6, can't be used by this strict workflow.) S3 wrote nothing."
        + ("\n" + "\n".join(_old) if _old else "")
    )

# ============================================================
# RESOLVE THE EXACT PROBE CROP ORIGIN
# ============================================================

# Source of truth: this recording's recording_crop_timing.npz (1.1). A crop or preprocessing
# still in memory from another recording / setting stops the cell instead of being used.
probe_crop_start_frame, probe_crop_start_s = nam.resolve_probe_crop(config, globals())

# Length must match what Kilosort reads: the saved S2 folder. Before S2 exists,
# the S1 recording in memory (same settings) is used.
si_save_path = nam.preprocessed_folder / config.get("processed_file_name", "si_preprocessed")
s2_stamp_path = si_save_path / nam.SETTINGS_STAMP_NAME
if si_save_path.exists():
    if not s2_stamp_path.exists():
        raise FileNotFoundError(
            f"The spike-band folder has no settings file ({s2_stamp_path}), so its crop can't be checked.\n"
            "It was made by an older notebook version or is half-written. Move or delete it, then rerun S2."
        )
    rec_for_length = si.load(si_save_path)
    s2_settings = json.loads(s2_stamp_path.read_text())
    timestamp_settings = {k: s2_settings[k] for k in nam.SHARED_SETTINGS_KEYS}
    length_source = f"S2 folder {si_save_path}"
    if int(s2_settings["probe_crop_start_frame"]) != probe_crop_start_frame:
        raise ValueError(
            f"The saved S2 folder was cropped at frame {s2_settings['probe_crop_start_frame']:,}, "
            f"but the crop info says {probe_crop_start_frame:,}. Rerun 1.1 → S1 → S2, or restore the folder."
        )
    print("Using the saved spike-band folder (S2) for length and settings.")
elif "rec_step6_spike" in globals() and "shared_settings" in globals():
    rec_for_length = rec_step6_spike
    timestamp_settings = {k: shared_settings[k] for k in nam.SHARED_SETTINGS_KEYS}
    length_source = "S1 recording in memory (S2 not saved yet)"
    print("S2 not saved yet: using the S1 recording in memory.")
else:
    raise RuntimeError("Run S2 first (or 1.1 and S1 in this session).")

probe_sampling_rate_hz = float(rec_for_length.get_sampling_frequency())
expected_n_frames = int(rec_for_length.get_num_frames())

expected_crop_start_frame = int(round(probe_crop_start_s * probe_sampling_rate_hz))
if expected_crop_start_frame != probe_crop_start_frame:
    raise ValueError(
        "Saved crop time/frame metadata are inconsistent:\n"
        f"stored frame = {probe_crop_start_frame}\n"
        f"time-derived frame = {expected_crop_start_frame}"
    )

# ============================================================
# VALIDATE THE ORIGINAL FILES AND THE REQUESTED CROP
# ============================================================

# Identity of the two source files (size, modification time, sampled SHA-1), recorded with the
# outputs; S4 and A1 refuse the outputs if either file changes later
# (set nam.STRICT_SOURCE_HASH = True before this cell to add full SHA-256 checksums: slower)
source_identity = {"timestamps": nam.source_file_identity(original_timestamps_path),
                   "sample_numbers": nam.source_file_identity(original_sample_numbers_path)}

timestamps_full = np.load(original_timestamps_path, mmap_mode="r")
sample_numbers_full = np.load(original_sample_numbers_path, mmap_mode="r")

_problems = []
if timestamps_full.ndim != 1 or timestamps_full.dtype.kind != "f":
    _problems.append(f"timestamps.npy: shape {timestamps_full.shape}, dtype {timestamps_full.dtype} "
                     "(expected 1-D floating-point seconds)")
if sample_numbers_full.ndim != 1 or sample_numbers_full.dtype.kind not in "iu":
    _problems.append(f"sample_numbers.npy: shape {sample_numbers_full.shape}, dtype {sample_numbers_full.dtype} "
                     "(expected 1-D integers)")
if len(timestamps_full) != len(sample_numbers_full):
    _problems.append(f"lengths differ: timestamps.npy {len(timestamps_full):,}, "
                     f"sample_numbers.npy {len(sample_numbers_full):,}")
end_frame = probe_crop_start_frame + expected_n_frames
if probe_crop_start_frame < 0 or end_frame > len(timestamps_full):
    _problems.append(f"the requested range {probe_crop_start_frame:,}-{end_frame:,} (crop start + "
                     f"{expected_n_frames:,} frames from the {length_source}) is outside the "
                     f"{len(timestamps_full):,} recorded samples")
if _problems:
    raise ValueError("S3 inputs don't fit together (nothing written):\n  " + "\n  ".join(_problems))

timestamps_trimmed = np.asarray(timestamps_full[probe_crop_start_frame:end_frame])
sample_numbers_trimmed = np.asarray(sample_numbers_full[probe_crop_start_frame:end_frame])

# ============================================================
# CLOCK CHECK BEFORE WRITING: forward gaps (lost samples) are reported; NaN/inf timestamps,
# repeated or backward sample numbers and backward timestamp steps beyond Open Ephys's
# synchronization corrections (nam.SYNC_CORRECTION_MAX_S) stop the cell, so S4 can't sort
# a recording whose clock is broken in the retained part.
# ============================================================

gap_report = nam.check_timestamp_gaps(timestamps_trimmed, sample_numbers_trimmed, probe_sampling_rate_hz)
if not gap_report["clock_ok_for_sorting"]:
    raise ValueError(
        "The retained part of the probe's timestamps / sample numbers is not usable (nothing written):\n"
        f"  NaN/inf timestamps:        {gap_report['n_nonfinite_timestamps']}\n"
        f"  repeated sample numbers:   {gap_report['n_repeated_samples']}\n"
        f"  backward sample numbers:   {gap_report['n_backward_jumps']}\n"
        f"  backward timestamp steps:  {gap_report['n_timestamp_backward_steps']} "
        f"(beyond {1e3 * nam.SYNC_CORRECTION_MAX_S:g} ms Open Ephys corrections)\n"
        f"  unexplained timestamp jumps: {gap_report['n_unexplained_timestamp_jumps']} "
        f"(not matched by missing sample numbers; e.g. {gap_report['first_unexplained_timestamp_jumps'][:2]})\n"
        f"First irregular samples: {gap_report.get('first_sample_number_gaps', [])[:3]} "
        f"{gap_report.get('first_nonfinite_timestamps', [])[:3]}\n"
        "Check the original Open Ephys files; a different REMOVE_FIRST_MINUTES only helps if the problem "
        "lies in the discarded part."
    )

# ============================================================
# SAVE (an older, inferred version of these files is replaced, with a note)
# ============================================================

if gap_report_path.exists():
    try:
        if json.loads(gap_report_path.read_text()).get("sample_numbers_inferred"):
            print("Note: replacing S3 outputs from an older notebook version whose sample numbers were "
                  "INFERRED; the new files are read from sample_numbers.npy.")
    except Exception:
        pass

import os
gap_report["probe_crop_start_frame"] = probe_crop_start_frame
gap_report["sample_numbers_source"] = f"read: {original_sample_numbers_path}"
gap_report["timestamps_source"] = f"read: {original_timestamps_path}"
gap_report["sample_numbers_inferred"] = False

# Staged write: new files go to temporary names first; the old settings file (the marker S4/A1
# need) is removed, the new files are moved into place, and the settings file is written LAST.
# An interrupted save therefore leaves no settings file, and S4/A1 refuse the outputs.
nam.preprocessed_folder.mkdir(parents=True, exist_ok=True)
_tmp_ts = timestamps_trimmed_path.with_name("timestamps_trimmed.tmp.npy")
_tmp_sn = sample_numbers_trimmed_path.with_name("sample_numbers_trimmed.tmp.npy")
_tmp_rep = gap_report_path.with_name("timestamp_gap_report.tmp.json")
np.save(_tmp_ts, timestamps_trimmed)
np.save(_tmp_sn, sample_numbers_trimmed)
_tmp_rep.write_text(json.dumps(gap_report, indent=2))
# Checksums of S3's own outputs (S4 / A1 recompute them: a later edit of these files is refused)
output_sha256 = {timestamps_trimmed_path.name: nam.file_sha256(_tmp_ts),
                 sample_numbers_trimmed_path.name: nam.file_sha256(_tmp_sn),
                 gap_report_path.name: nam.file_sha256(_tmp_rep)}
ts_settings_path = timestamps_trimmed_path.with_name(timestamps_trimmed_path.name + ".settings.json")
ts_settings_path.unlink(missing_ok=True)
try:
    os.replace(_tmp_ts, timestamps_trimmed_path)
    os.replace(_tmp_sn, sample_numbers_trimmed_path)
    os.replace(_tmp_rep, gap_report_path)
except PermissionError as exc:
    raise PermissionError(
        f"Windows won't replace an S3 output that is still open (e.g. memory-mapped by an earlier cell): {exc}\n"
        "Restart the kernel, run 0.1 and 0.2, then S3 again. The old outputs have no settings file now, "
        "so S4/A1 won't use them."
    ) from exc
_stamp = nam.write_settings_stamp(timestamps_trimmed_path, {
    **timestamp_settings,
    "s3_provenance": {
        "timestamps_source": str(original_timestamps_path),
        "sample_numbers_source": str(original_sample_numbers_path),
        "sample_numbers": "read",
        "source_n_samples": int(len(timestamps_full)),
        "source_identity": source_identity,
        "trimmed_range": [int(probe_crop_start_frame), int(end_frame)],
        "length_from": length_source,
    },
    "s3_output_sha256": output_sha256,
})

print("Probe crop start frame:", probe_crop_start_frame)
print("Probe crop start time (s):", probe_crop_start_s)
print("Probe sampling rate (Hz):", probe_sampling_rate_hz)
print("Expected cropped frames:", expected_n_frames)
print("Trimmed timestamp frames:", len(timestamps_trimmed))
print("Trimmed sample-number frames:", len(sample_numbers_trimmed))
print("Sample numbers: READ from", original_sample_numbers_path)
print("Saved:", timestamps_trimmed_path)
print("Saved:", sample_numbers_trimmed_path)
print("Settings saved (with source-file identity):", _stamp.name)

# ============================================================
# GAP REPORT (computed above): dropped samples and synchronization corrections
# ============================================================

if gap_report.get("n_sync_corrections"):
    print(f"Open Ephys synchronization corrections: {gap_report['n_sync_corrections']} "
          f"(largest {1e6 * gap_report['largest_sync_correction_s']:.1f} us; not counted as gaps).")
if gap_report["ok"]:
    print("Gap check: no dropped or repeated samples.")
else:
    print(
        "[warning] Gap check found irregular samples:\n"
        f"  forward gaps:       {gap_report['n_forward_gaps']} "
        f"(missing samples in total: {gap_report['missing_samples_total']})\n"
        f"  repeated samples:   {gap_report['n_repeated_samples']}\n"
        f"  backward jumps:     {gap_report['n_backward_jumps']}\n"
        f"  timestamp gaps:     {gap_report['n_timestamp_gaps']}\n"
        f"  NaN/inf timestamps: {gap_report['n_nonfinite_timestamps']}\n"
        "  Kilosort's sample indices are unaffected, but sample / fs is no longer exact "
        "acquisition time. Use timestamps_trimmed.npy (as A1 does) for real times."
    )
if gap_report.get("timestamps_are_sample_counts"):
    print("Note: timestamps.npy holds sample counts, not seconds (older Open Ephys format).")
print("Gap report:", gap_report_path)

### S4 Run Kilosort4
1. Reloads the saved spike-band recording from S2 (so Kilosort sorts exactly what's on disk).
2. Sanity checks: one segment, reasonable length, valid probe geometry, finite traces. **S3 must have run:** its trimmed timestamps must exist, be read from Open Ephys's own `sample_numbers.npy` (not inferred), carry the same crop and channels as this S2 folder, match its frame count, and S3's report must confirm a usable clock (`clock_ok_for_sorting`); otherwise the cell stops and says what to rerun.
3. Checks that Docker is running and the Kilosort image is on this machine **before** starting, so it fails in seconds rather than mid-run.
4. Runs Kilosort4 in Docker.

Refuses to run if a previous `sorter_output` already exists, so old sorts are never overwritten. It also stops on a leftover Kilosort folder from a failed run, which SpikeInterface would otherwise reject only after Docker has started.

**Edit:**
- `KILOSORT4_DOCKER_IMAGE`: defaults to the official image **pinned** to Kilosort 4.0.38, `spikeinterface/kilosort4-base:4.0.38_cuda-12.0.0`, so every run uses the same Kilosort. Avoid `:latest`: it changes over time, and Kilosort ≥ 4.1.2 needs SpikeInterface ≥ 0.103.2 (S4 stops with an explanation if they don't match). The run record stores the image ID and digest. See *Docker setup* below.
- `KS_NBLOCKS`: drift correction. `1` = rigid (whole probe shifts together, the Kilosort default). Try `5` on a long single-shank recording if the drift plot in Kilosort's output shows different depths moving differently. `0` turns drift correction off.
- `kilosort4_params`: `do_CAR` is off because S1 already did CMR. Kilosort's own preprocessing stays **on** because it includes whitening, which Kilosort needs. That preprocessing also applies Kilosort's own 300 Hz high-pass (3rd-order Butterworth, parameter `highpass_cutoff`) on top of S1's 300 Hz high-pass, so the effective high-pass is steeper than either alone. This is a deliberate, documented choice; skipping Kilosort preprocessing is not the way around it, because SpikeInterface then also replaces whitening with an identity matrix.

After sorting it writes `nam_kilosort_run_record.json` next to `sorter_output`: the Docker image (name and ID), Kilosort parameters, SpikeInterface version, and the input folder with its settings.

**Makes:** `kilosort4_sorting`, the Kilosort output folder, and the run record.

---
#### Docker setup (one time per machine)
SpikeInterface runs Kilosort4 inside a Docker container, so Kilosort, PyTorch and CUDA don't have to be installed in your own Python environment. The image has to be on the machine that runs this notebook.

1. **Install Docker Desktop for Windows** from docker.com. During setup keep *Use WSL 2* ticked. Restart, then open Docker Desktop and leave it running.
2. **Check the GPU driver.** Install a current NVIDIA driver. Docker Desktop with WSL 2 passes the GPU into containers automatically; no CUDA install is needed on Windows.
3. **Install the Python Docker client** in the same environment as this notebook (in Anaconda Prompt, with that environment activated):
   ```
   pip install docker
   ```
4. **Download the Kilosort4 image** (several GB, takes a while):
   ```
   docker pull spikeinterface/kilosort4-base:4.0.38_cuda-12.0.0
   ```
5. **Test that the container sees the GPU:**
   ```
   docker run --rm --gpus all spikeinterface/kilosort4-base:4.0.38_cuda-12.0.0 nvidia-smi
   ```
   You should see your GPU listed. If you get an error, the GPU isn't reaching Docker: update the NVIDIA driver and restart Docker Desktop.
6. **Confirm the image is there:**
   ```
   docker images
   ```
   `spikeinterface/kilosort4-base` should appear in the list.

**Using your own custom image instead** (e.g. the old `kilosort4-si:0.103`): it must already exist on this machine (check with `docker images`), because a locally built image can't be downloaded. Set `KILOSORT4_DOCKER_IMAGE` to its exact name and tag. Alternatively, give the official image that name with `docker tag spikeinterface/kilosort4-base:4.0.38_cuda-12.0.0 kilosort4-si:0.103`.

When it runs, SpikeInterface installs its own matching version inside the container automatically, so the first run takes a few extra minutes.

In [ ]:
# ==========================================================================
# [S4] Load saved spike-band recording and run Kilosort4
# Needs: S2 on disk, S3 (trimmed timestamps)
# Makes: kilosort4_sorting, <kilosort folder>/sorter_output/,
#        <kilosort folder>/nam_kilosort_run_record.json
# ==========================================================================

import json
import Neuropixel_analysis_module as nam
import spikeinterface.full as si
import numpy as np
from pathlib import Path

config = nam.load_experiment_config(config_path)
nam.define_analysis_paths_from_config(config_path)

processed_file_name = config.get(
    "processed_file_name",
    "si_preprocessed",
)

si_save_path = (
    nam.preprocessed_folder
    / processed_file_name
)

if not si_save_path.exists():
    raise FileNotFoundError(
        "The exact saved preprocessing output was not found. "
        "Run S2 first:\n"
        f"{si_save_path}"
    )

print("Loading preprocessed recording:")
print(si_save_path)

reloaded_rec = si.load(si_save_path)

sampling_frequency = float(
    reloaded_rec.get_sampling_frequency()
)
n_channels = int(
    reloaded_rec.get_num_channels()
)
n_frames = int(
    reloaded_rec.get_num_frames()
)
n_segments = int(
    reloaded_rec.get_num_segments()
)

print(reloaded_rec)
print("Sampling frequency:", sampling_frequency)
print("Channels:", n_channels)
print("Frames:", n_frames)
print("Segments:", n_segments)

if n_segments != 1:
    raise ValueError(
        "Kilosort4 requires one continuous segment; "
        f"found {n_segments}."
    )

if n_frames <= 1000:
    raise ValueError(
        f"The saved recording is unexpectedly short: {n_frames} frames."
    )

try:
    channel_locations = np.asarray(
        reloaded_rec.get_channel_locations(),
        dtype=float,
    )
except Exception as exc:
    raise RuntimeError(
        "The saved recording has no usable probe geometry."
    ) from exc

if (
    channel_locations.ndim != 2
    or len(channel_locations) != n_channels
    or not np.all(np.isfinite(channel_locations))
):
    raise ValueError(
        "Invalid channel-location array: "
        f"shape={channel_locations.shape}, "
        f"expected rows={n_channels}."
    )

print("Channel-location shape:", channel_locations.shape)

# Prerequisite: S3's trimmed timestamps, READ from Open Ephys files, with the same crop and
# channels as this S2 folder and its frame count (raises with what to rerun otherwise)
_ts_settings, _gap_report = nam.verify_trimmed_timestamps(nam.preprocessed_folder, si_save_path)
if int(json.loads((si_save_path / nam.SETTINGS_STAMP_NAME).read_text())["raw_n_frames"]) != n_frames:
    raise ValueError("The S2 folder's settings file doesn't match its frame count. Rerun S2.")
print("Trimmed timestamps (S3): match this S2 folder; sample numbers read from",
      _ts_settings["s3_provenance"]["sample_numbers_source"])

trace_test = reloaded_rec.get_traces(
    start_frame=0,
    end_frame=1000,
)

print("Trace test:", trace_test.shape)
print("Trace dtype:", trace_test.dtype)
print("Finite trace values:", np.all(np.isfinite(trace_test)))

if not np.all(np.isfinite(trace_test)):
    raise ValueError(
        "The saved recording contains non-finite values in the trace test."
    )

# ============================================================
# KILOSORT SETTINGS
# ============================================================

# Official SpikeInterface image, PINNED to Kilosort 4.0.38 (see "Docker setup" above): "latest" can change,
# and Kilosort >= 4.1.2 needs SpikeInterface >= 0.103.2 (0.103.0 fails to unpack its run parameters).
# For a custom local image, put its exact name:tag here, e.g. "kilosort4-si:0.103".
KILOSORT4_DOCKER_IMAGE = "spikeinterface/kilosort4-base:4.0.38_cuda-12.0.0"

# Drift correction: 1 = rigid (default), 5 = non-rigid for long single-shank drift, 0 = off
KS_NBLOCKS = 1

kilosort4_params = {
    "do_CAR": False,                       # S1 already did global median CMR
    "skip_kilosort_preprocessing": False,  # keep on: includes whitening
    "nblocks": KS_NBLOCKS,
}

# Kilosort >= 4.1.2 returns an extra run parameter that SpikeInterface < 0.103.2 can't unpack
import re as _re
from packaging.version import Version as _V
_m = _re.search(r"kilosort4-base:(\d+\.\d+\.\d+)", KILOSORT4_DOCKER_IMAGE)
if _m and _V(_m.group(1)) >= _V("4.1.2") and _V(si.__version__) < _V("0.103.2"):
    raise RuntimeError(
        f"{KILOSORT4_DOCKER_IMAGE} has Kilosort {_m.group(1)}, which needs SpikeInterface >= 0.103.2 "
        f"(this environment: {si.__version__}). Use 'spikeinterface/kilosort4-base:4.0.38_cuda-12.0.0' or upgrade SpikeInterface."
    )
if KILOSORT4_DOCKER_IMAGE.endswith(":latest"):
    print("[warning] ':latest' can point to a different Kilosort version later; pin a version tag for reproducible sorts.")

# ============================================================
# CHECK DOCKER AND THE IMAGE BEFORE STARTING
# ============================================================

try:
    import docker
    from docker.errors import ImageNotFound
except ImportError as exc:
    raise ImportError(
        "The Python 'docker' package is missing. In this environment run:\n"
        "    pip install docker"
    ) from exc

try:
    docker_client = docker.from_env()
    docker_client.ping()
except Exception as exc:
    raise RuntimeError(
        "Docker is not reachable. Open Docker Desktop, wait until it says "
        "'Engine running', then rerun this cell."
    ) from exc

try:
    docker_image_obj = docker_client.images.get(KILOSORT4_DOCKER_IMAGE)
    print("Docker image found locally:", KILOSORT4_DOCKER_IMAGE)
except ImageNotFound:
    raise RuntimeError(
        f"Docker image not found on this machine: {KILOSORT4_DOCKER_IMAGE}\n"
        "Download it in a terminal with:\n"
        f"    docker pull {KILOSORT4_DOCKER_IMAGE}\n"
        "(A custom image like kilosort4-si:0.103 can't be pulled; it must be built "
        "on this machine or the official image tagged with that name. See Docker setup.)"
    )

sorter_output_path = (
    Path(nam.kilosort_folder)
    / "sorter_output"
)

if (
    sorter_output_path.exists()
    and any(sorter_output_path.iterdir())
):
    raise FileExistsError(
        "An existing Kilosort sorter_output folder was found:\n"
        f"{sorter_output_path}\n"
        "Move or rename it before starting another run."
    )
if Path(nam.kilosort_folder).exists() and any(Path(nam.kilosort_folder).iterdir()):
    raise FileExistsError(
        "An existing Kilosort folder was found (probably left by an earlier failed run):\n"
        f"{nam.kilosort_folder}\n"
        "SpikeInterface won't sort into an existing folder. Move or rename it before starting another run."
    )

print("Kilosort4 input:", si_save_path)
print("Kilosort4 output:", nam.kilosort_folder)
print("Kilosort4 parameters:", kilosort4_params)

kilosort4_sorting = si.run_sorter(
    sorter_name="kilosort4",
    recording=reloaded_rec,
    folder=nam.kilosort_folder,
    verbose=True,
    raise_error=True,
    remove_existing_folder=False,
    docker_image=KILOSORT4_DOCKER_IMAGE,
    **kilosort4_params,
)

print("Kilosort4 completed:")
print(kilosort4_sorting)
print(
    "Units detected:",
    len(kilosort4_sorting.get_unit_ids()),
)
print("Sorter output:", sorter_output_path)

# ============================================================
# RUN RECORD: what was sorted, with which image and parameters
# ============================================================

import datetime

input_settings_path = si_save_path / nam.SETTINGS_STAMP_NAME
run_record = {
    "finished_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "docker_image": KILOSORT4_DOCKER_IMAGE,
    "docker_image_id": docker_image_obj.id,
    "docker_image_digests": docker_image_obj.attrs.get("RepoDigests", []),
    "spikeinterface_version": si.__version__,
    "kilosort4_params": kilosort4_params,
    "input_recording": str(si_save_path),
    "input_n_frames": n_frames,
    "input_n_channels": n_channels,
    "input_sampling_frequency_hz": sampling_frequency,
    "input_settings": json.loads(input_settings_path.read_text()),
    "input_timestamps_settings": _ts_settings,
    "n_units": int(len(kilosort4_sorting.get_unit_ids())),
    "sort_fingerprint": nam.sort_fingerprint(sorter_output_path),
}
run_record_path = Path(nam.kilosort_folder) / "nam_kilosort_run_record.json"
with open(run_record_path, "w") as f:
    json.dump(nam._jsonable(run_record), f, indent=2)
print("Run record saved:", run_record_path)

# Optional parameter inspection:
# params = si.get_default_sorter_params(
#     sorter_name_or_class="kilosort4"
# )
# print("Parameters:\n", params)
# desc = si.get_sorter_params_description(
#     sorter_name_or_class="kilosort4"
# )
# print("Descriptions:\n", desc)

### S5 Register Phy curation (optional)
Run this only after **merging or splitting clusters in Phy**. Relabelling alone (good / mua / noise) doesn't need it; C5 reads your Phy labels directly.

Merges and splits rewrite `spike_clusters.npy`. Every later cell checks that `sorter_output` is the version Kilosort wrote, so they stop until the curated version is registered. This cell:
- checks that `spike_times.npy` is still exactly what Kilosort produced (Phy never changes spike times), and that `spike_clusters.npy` has one integer per spike;
- leaves S4's run record untouched, and writes a separate `nam_phy_curation_record.json` linking the curated version to it.

After that, rerun C1–C5. They build fresh outputs for the curated sort: a new `si_analyzer_<sort id>/` and a new CellExplorer folder. Older ones stay on disk. If you curate again later, run S5 again.

**Edit:** nothing.

**Makes:** `nam_phy_curation_record.json` in the Kilosort folder.


In [ ]:
# ==========================================================================
# [S5] Register a Phy-curated version of the Kilosort output (after merges / splits)
# Needs: S4 (run record); Phy edits saved in sorter_output
# Makes: <kilosort folder>/nam_phy_curation_record.json
# ==========================================================================

from pathlib import Path
import Neuropixel_analysis_module as nam

config = nam.load_experiment_config(config_path)
nam.define_analysis_paths_from_config(config_path)

phy = nam.register_phy_curation(nam.kilosort_folder)

print("Phy curation registered:", Path(nam.kilosort_folder) / nam.PHY_RECORD_NAME)
print(f"Clusters: {phy['n_clusters_original']} from Kilosort -> {phy['n_clusters']} after Phy")
print("Spike times unchanged; S4's run record kept as the original. Now rerun C1–C5.")


---
## 3 · Curation after Kilosort (optional)
Run after **S4**. Nothing in the LFP branch depends on it. Each cell checks that its inputs exist and stops with a clear message if not.

| Cell | What it does | Needs |
|---|---|---|
| **C1** | Sorting analyzer: waveforms, templates, quality and template metrics | S2, S4 |
| **C2** | UnitRefine labels (noise / SUA / MUA), keyed by real Kilosort cluster IDs | C1 |
| **C3** | 16-bit copy of the spike-band data for Bombcell and CellExplorer (MATLAB) | S2, S4 |
| **C4** | Writes the Bombcell and CellExplorer MATLAB scripts; you run them in MATLAB | C3 |
| **C5** | Summary table: one row per Kilosort cluster, merging whatever has been run | S4 (+ any of C2, C4) |

You can skip any of C1–C4. C5 uses whatever results exist. A unit counts as **consensus good** only if every tool that was run calls it good.

### C1 Sorting analyzer
Builds a SpikeInterface sorting analyzer from the Kilosort output and the saved spike-band recording (the exact data Kilosort sorted). It computes what UnitRefine needs: noise levels, waveforms, templates, spike amplitudes and locations, correlograms, principal components, quality metrics and template metrics (including the multi-channel ones).

Unit IDs are Kilosort's own cluster IDs, read straight from `sorter_output`. Before building anything, it checks from S4's run record that the current S2 folder is the exact input Kilosort sorted (same path and settings) and that `sorter_output` is unchanged. A re-saved S2 with different preprocessing would give waveforms that don't match the spikes.

The analyzer folder also gets a settings file that identifies the Kilosort output it was built from (a fingerprint of `spike_times.npy` and `spike_clusters.npy`); C2 checks it before labelling. If the folder for the current sort already exists and matches, the cell loads it and only computes extensions that are missing.

This is the slowest curation step: expect several minutes to an hour, depending on recording length and unit count.

**Edit:** `ANALYZER_N_JOBS` (parallel workers; `-1` = all cores).

Each sort gets its own analyzer folder, `si_analyzer_<sort id>/`, named after its spike times and cluster assignments. A new Kilosort run or a registered Phy curation (S5) therefore builds a fresh analyzer, and older ones stay on disk.

**Makes:** `sorting_analyzer`, saved in `si_analyzer_<sort id>/` in the data folder.

In [ ]:
# ==========================================================================
# [C1] Sorting analyzer from the Kilosort output
# Needs: S2 (spike-band folder), S4 (Kilosort sorter_output)
# Makes: sorting_analyzer, <data folder>/si_analyzer_<sort id>/
# ==========================================================================

import numpy as np
from pathlib import Path
import spikeinterface.full as si
import Neuropixel_analysis_module as nam

# ----- settings -----
ANALYZER_N_JOBS = -1
# --------------------

config = nam.load_experiment_config(config_path)
nam.define_analysis_paths_from_config(config_path)

si_save_path = nam.preprocessed_folder / config.get("processed_file_name", "si_preprocessed")
sorter_output = Path(nam.kilosort_folder) / "sorter_output"

for path, made_by in [
    (si_save_path, "S2"),
    (sorter_output / "spike_times.npy", "S4"),
    (sorter_output / "spike_clusters.npy", "S4"),
]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}\nRun {made_by} first.")

# The S2 folder must be the exact input of the current Kilosort run (S4's run record):
# a re-saved S2 with other preprocessing would give waveforms that don't match the spikes
nam.verify_sort_input(nam.kilosort_folder, si_save_path)
print("Provenance OK: S2 folder is the one Kilosort sorted, and sorter_output is unchanged "
      "(or a registered Phy curation of it).")

# One analyzer folder per sort (spike times + cluster assignments), so a Phy-curated
# version or a new Kilosort run gets a fresh analyzer and older ones stay untouched
analyzer_folder = Path(nam.si_folder).with_name(f"si_analyzer_{nam.sort_id(sorter_output)}")


recording = si.load(si_save_path)
# Units straight from spike_times.npy + spike_clusters.npy. (SpikeInterface's Phy reader
# inner-joins all cluster_*.tsv files and drops clusters missing from any of them, e.g. a
# merged Phy cluster that cluster_KSLabel.tsv doesn't list, together with its spikes.)
# Validated BEFORE the integer conversion: NaN/inf, fractional or negative spike times,
# spike times past the end of the recording, and bad cluster IDs stop the cell here
# instead of being silently changed by the cast.
n_frames = int(recording.get_num_frames())
_st = nam.load_spike_samples(sorter_output / "spike_times.npy", n_frames=n_frames)
_clu = nam.load_spike_clusters(sorter_output / "spike_clusters.npy", n_spikes=_st.size)
_from = getattr(si.NumpySorting, "from_samples_and_labels", None) or si.NumpySorting.from_times_labels
sorting = _from([_st], [_clu], recording.get_sampling_frequency())
_n_in_sorting = sum(sorting.get_unit_spike_train(u).size for u in sorting.unit_ids)
if _n_in_sorting != _st.size:
    raise ValueError(f"Only {_n_in_sorting:,} of {_st.size:,} spikes made it into the sorting.")

analyzer_settings = {
    "recording": str(si_save_path),
    "recording_n_frames": n_frames,
    "sorter_output": str(sorter_output),
    **nam.sort_fingerprint(sorter_output),   # spike_times_sha1, spike_clusters_sha1
    "n_units": int(len(sorting.unit_ids)),
    "sparse": True,
}

print(f"Units (non-empty): {len(sorting.unit_ids)}  |  first IDs: {list(sorting.unit_ids[:10])}")

if analyzer_folder.exists():
    status = nam.check_settings_stamp(analyzer_folder, analyzer_settings)  # raises if Kilosort changed
    if status == "no_stamp":
        raise RuntimeError(
            "An analyzer folder from an older run exists and can't be checked against this "
            f"Kilosort output:\n{analyzer_folder}\nMove or delete it, then rerun this cell."
        )
    sorting_analyzer = si.load_sorting_analyzer(analyzer_folder)
    print("Loaded existing analyzer (matches this Kilosort output):", analyzer_folder)
else:
    sorting_analyzer = si.create_sorting_analyzer(
        sorting=sorting,
        recording=recording,
        format="binary_folder",
        folder=analyzer_folder,
        sparse=True,
    )
    _stamp = nam.write_settings_stamp(analyzer_folder, analyzer_settings)
    print("Analyzer settings saved:", _stamp.name)
    print("Created analyzer:", analyzer_folder)

job_kwargs = dict(n_jobs=ANALYZER_N_JOBS, chunk_duration="1s", progress_bar=True)

# Extensions UnitRefine needs (SpikeInterface model-based curation tutorial)
extension_order = [
    ("random_spikes", {}),
    ("noise_levels", {}),
    ("waveforms", {}),
    ("templates", {}),
    ("spike_amplitudes", {}),
    ("spike_locations", {}),
    ("correlograms", {}),
    ("principal_components", {}),
    ("quality_metrics", {}),
    ("template_metrics", {"include_multi_channel_metrics": True}),
]

for name, params in extension_order:
    if sorting_analyzer.has_extension(name):
        print(f"  {name}: already computed")
        continue
    print(f"  computing {name} ...")
    sorting_analyzer.compute(name, **params, **job_kwargs)

qm = sorting_analyzer.get_extension("quality_metrics").get_data()
tm = sorting_analyzer.get_extension("template_metrics").get_data()
print(f"\nQuality metrics: {qm.shape[1]} columns for {qm.shape[0]} units")
print(f"Template metrics: {tm.shape[1]} columns for {tm.shape[0]} units")
print("Analyzer ready:", analyzer_folder)

### C2 UnitRefine
Runs the two pretrained UnitRefine models on the C1 analyzer: **noise vs neural**, then **SUA vs MUA**. Labels are saved with the real Kilosort cluster IDs (the old code numbered units 0, 1, 2 … by position, which didn't match Kilosort).

The first run downloads the models from Hugging Face, so it needs internet access, plus the `huggingface_hub` and `skops` packages (`pip install huggingface_hub skops`).

Before running, it checks that the C1 analyzer was built from the **current** Kilosort output. The result is tagged with that sort's fingerprint (`unit_refine_results.csv.settings.json`), so C5 rejects it if Kilosort is rerun later, even if the cluster IDs happen to match.

An old `unit_refine_results.csv` with the made-up IDs is overwritten here. Until then, C5 ignores it with a warning.

**Model-compatible environment:** the UnitRefine models were saved with scikit-learn 1.4.2. In a newer scikit-learn they still load but print `InconsistentVersionWarning`. Set `UNITREFINE_PYTHON` to the `python.exe` of a matching environment (e.g. `unitrefine_py3_12`, which also needs SpikeInterface 0.103.x, `huggingface_hub` and `skops`): C2 then runs the same labelling function in that environment on the analyzer C1 saved, checks that the unit IDs are exactly the analyzer's cluster IDs, and records the Python, scikit-learn and SpikeInterface versions in `unit_refine/unit_refine_run_info.json`. Find the path with `conda run -n unitrefine_py3_12 python -c "import sys; print(sys.executable)"`.

A run writes into a temporary folder first; the previous result and its tag are replaced only after the new run has succeeded and its unit IDs match, so a failed download or inference leaves the previous result usable.

**Edit:** `UNITREFINE_PYTHON` (optional).

**Makes:** `unit_refine_results.csv` in `unit_refine/` (columns `unit_id`, `Noise_Label`, `SUA_MUA_Label`, plus probabilities).

In [ ]:
# ==========================================================================
# [C2] UnitRefine labels with real Kilosort cluster IDs
# Needs: C1 (analyzer with quality + template metrics)
# Makes: <data folder>/unit_refine/unit_refine_results.csv (+ .settings.json with the sort fingerprint)
# ==========================================================================

from pathlib import Path
import Neuropixel_analysis_module as nam

import json
import subprocess
import pandas as pd

# ----- settings -----
# The UnitRefine models were saved with scikit-learn 1.4.2; this environment may have a newer one
# (then: InconsistentVersionWarning). To run C2 in a matching environment instead, give that
# environment's python.exe, e.g. r"C:\Users\<you>\miniforge3\envs\unitrefine_py3_12\python.exe"
# (find it with:  conda run -n unitrefine_py3_12 python -c "import sys; print(sys.executable)").
# It needs SpikeInterface 0.103.x (same analyzer format), huggingface_hub and skops. None = this kernel.
UNITREFINE_PYTHON = None
UNITREFINE_MODEL_SKLEARN = "1.4.2"
# --------------------

config = nam.load_experiment_config(config_path)
nam.define_analysis_paths_from_config(config_path)

sorter_output = Path(nam.kilosort_folder) / "sorter_output"
analyzer_folder = Path(nam.si_folder).with_name(f"si_analyzer_{nam.sort_id(sorter_output)}")   # same as C1
if not analyzer_folder.exists():
    raise FileNotFoundError(f"Missing analyzer: {analyzer_folder}\nRun C1 first.")

# The analyzer must come from the CURRENT Kilosort output (same spikes, not just same IDs)
current_sort = nam.sort_fingerprint(sorter_output)
analyzer_stamp = analyzer_folder / nam.SETTINGS_STAMP_NAME
if not analyzer_stamp.exists():
    raise RuntimeError(f"The analyzer has no settings file, so its sort can't be checked. Rerun C1.\n{analyzer_folder}")
analyzer_info = json.loads(analyzer_stamp.read_text())
if {k: analyzer_info.get(k) for k in current_sort} != current_sort:
    raise RuntimeError(
        "The analyzer was built from a different Kilosort run than the one in sorter_output now. "
        "Rerun C1, then C2."
    )

if not UNITREFINE_PYTHON:
    for package in ("huggingface_hub", "skops"):
        try:
            __import__(package)
        except ImportError as exc:
            raise ImportError(
                f"UnitRefine needs the '{package}' package. In this environment run:\n"
                "    pip install huggingface_hub skops"
            ) from exc
elif not Path(UNITREFINE_PYTHON).is_file():
    raise FileNotFoundError(f"UNITREFINE_PYTHON not found: {UNITREFINE_PYTHON}")

import os
import shutil
ur_csv = Path(nam.unit_refine_folder) / "unit_refine_results.csv"
ur_stamp = ur_csv.with_name(ur_csv.name + ".settings.json")
# Run into a temporary folder; the previous result and its tag stay untouched until this run
# has succeeded and its unit IDs are checked
ur_pending = Path(nam.unit_refine_folder) / "_pending_run"
shutil.rmtree(ur_pending, ignore_errors=True)
ur_pending.mkdir(parents=True)

if UNITREFINE_PYTHON:
    # Same function (same cluster-ID handling), run by the other environment's Python on the
    # analyzer saved by C1; this kernel only reads the CSV back.
    _code = (
        "import sys, json\n"
        "sys.path.insert(0, sys.argv[1])\n"
        "import Neuropixel_analysis_module as nam, sklearn, spikeinterface\n"
        "nam.apply_unit_refinement(sys.argv[2], sys.argv[3])\n"
        "print('UNITREFINE_VERSIONS ' + json.dumps({'python': sys.version.split()[0], "
        "'sklearn': sklearn.__version__, 'spikeinterface': spikeinterface.__version__}))\n"
    )
    _run = subprocess.run(
        [UNITREFINE_PYTHON, "-c", _code, str(Path(nam.__file__).resolve().parent),
         str(analyzer_folder), str(ur_pending)],
        capture_output=True, text=True,
    )
    print(_run.stdout[-3000:])
    if _run.returncode != 0 or "UNITREFINE_VERSIONS " not in _run.stdout:
        raise RuntimeError(f"UnitRefine failed in {UNITREFINE_PYTHON}:\n{_run.stderr[-3000:]}")
    ur_versions = json.loads(_run.stdout.split("UNITREFINE_VERSIONS ")[-1].splitlines()[0])
    ur_versions["python_executable"] = UNITREFINE_PYTHON
    unit_refine_results = pd.read_csv(ur_pending / "unit_refine_results.csv")
else:
    unit_refine_results = nam.apply_unit_refinement(analyzer_folder, ur_pending)
    import sys, sklearn, spikeinterface
    ur_versions = {"python": sys.version.split()[0], "sklearn": sklearn.__version__,
                   "spikeinterface": spikeinterface.__version__, "python_executable": sys.executable}

# Unit IDs must be exactly the analyzer's (= Kilosort's) cluster IDs
import spikeinterface.full as si
_analyzer_ids = sorted(int(u) for u in si.load_sorting_analyzer(analyzer_folder, load_extensions=False).unit_ids)
if sorted(unit_refine_results["unit_id"].astype(int)) != _analyzer_ids:
    raise ValueError("UnitRefine unit IDs don't match the analyzer's cluster IDs; result not tagged.")

# Success: tag the new result in the temporary folder, then move result and tag into place.
# Between the two moves the old tag sits next to the new result; it only passes C5 if it
# names the current sort, i.e. the sort the new result belongs to as well.
_pending_csv = ur_pending / "unit_refine_results.csv"
nam.write_settings_stamp(_pending_csv, current_sort)
os.replace(_pending_csv, ur_csv)
os.replace(_pending_csv.with_name(_pending_csv.name + ".settings.json"), ur_stamp)
shutil.rmtree(ur_pending, ignore_errors=True)
(Path(nam.unit_refine_folder) / "unit_refine_run_info.json").write_text(json.dumps(
    {**ur_versions, "model_saved_with_sklearn": UNITREFINE_MODEL_SKLEARN}, indent=2))
print("UnitRefine ran with:", ur_versions)
if ur_versions["sklearn"] != UNITREFINE_MODEL_SKLEARN:
    print(f"[note] scikit-learn {ur_versions['sklearn']} differs from the models' {UNITREFINE_MODEL_SKLEARN}. "
          "Set UNITREFINE_PYTHON to a matching environment (e.g. unitrefine_py3_12) to avoid version effects.")

# Tag the result with the sort it belongs to; C5 refuses it if Kilosort is rerun later
print("UnitRefine result tagged with this sort:", ur_stamp.name)

print("\nNoise / neural:")
print(unit_refine_results["Noise_Label"].value_counts().to_string())
print("\nSUA / MUA:")
print(unit_refine_results["SUA_MUA_Label"].value_counts().to_string())

### C3 16-bit data file for MATLAB tools
Bombcell and CellExplorer read raw waveforms from a plain 16-bit binary file (`.dat`, channels interleaved). Kilosort ran from SpikeInterface's float32 folder, so there is no such file yet. This cell writes one from the S2 folder. S2 holds the filtered, referenced data as float32; the `.dat` is a **rounded 16-bit approximation** of it, in the original ADC units (multiply by `gain_to_uV` for µV). Rounding changes each value by at most 0.5 ADC unit (about 0.1 µV at a 0.195 µV/bit gain), far below the noise, but it is not bit-identical to what Kilosort sorted.

It first checks, from S4's run record, that the S2 folder is the one Kilosort sorted.

**Safety checks:**
- **Before writing:** enough free disk space; channel order and positions match Kilosort's `channel_map.npy` / `channel_positions.npy` (so waveforms line up with Kilosort's templates); one gain for all channels.
- **While writing:** every value must fit in 16 bits. If one doesn't, the cell stops and deletes the partial file; nothing is silently clipped.
- **After writing:** reads back the start and end of the file and checks they equal the rounded source exactly.

It writes to a temporary name and renames only after all checks pass, so a half-written file is never mistaken for a finished one. A settings file next to it records the source; if the file already exists and matches, the cell skips writing.

**Size:** 2 bytes × channels × samples, about 80 GB per hour for 384 channels at 30 kHz.

**Edit:** `INT16_FILE_NAME` (optional).

**Makes:** `matlab_curation/spikeband_int16.dat` and `spikeband_int16.dat.settings.json` in the data folder.

In [ ]:
# ==========================================================================
# [C3] 16-bit binary copy of the spike-band recording (for Bombcell / CellExplorer)
# Needs: S2 (spike-band folder), S4 (Kilosort channel map / positions)
# Makes: <data folder>/matlab_curation/spikeband_int16.dat (+ .settings.json)
# ==========================================================================

import json
import shutil
import numpy as np
from pathlib import Path
import spikeinterface.full as si
import Neuropixel_analysis_module as nam

# ----- settings -----
INT16_FILE_NAME = "spikeband_int16.dat"
# --------------------

config = nam.load_experiment_config(config_path)
nam.define_analysis_paths_from_config(config_path)

si_save_path = nam.preprocessed_folder / config.get("processed_file_name", "si_preprocessed")
sorter_output = Path(nam.kilosort_folder) / "sorter_output"
matlab_folder = Path(nam.data_folder) / "matlab_curation"
dat_path = matlab_folder / INT16_FILE_NAME

for path, made_by in [
    (si_save_path, "S2"),
    (sorter_output / "channel_map.npy", "S4"),
    (sorter_output / "channel_positions.npy", "S4"),
]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}\nRun {made_by} first.")

# The S2 folder must be the exact input of the current Kilosort run (S4's run record):
# a re-saved S2 with other preprocessing would give waveforms that don't match the spikes
nam.verify_sort_input(nam.kilosort_folder, si_save_path)
print("Provenance OK: S2 folder is the one Kilosort sorted, and sorter_output is unchanged.")

source = si.load(si_save_path)
fs = float(source.get_sampling_frequency())
n_frames = int(source.get_num_frames())
n_channels = int(source.get_num_channels())
expected_bytes = n_frames * n_channels * 2

# ---------- channel order must match Kilosort ----------
ks_channel_map = np.load(sorter_output / "channel_map.npy").ravel().astype(int)
ks_positions = np.load(sorter_output / "channel_positions.npy")
rec_positions = np.asarray(source.get_channel_locations(), dtype=float)

if not np.array_equal(ks_channel_map, np.arange(n_channels)):
    raise ValueError(
        "Kilosort's channel_map.npy is not 0..N-1 for this recording "
        f"({len(ks_channel_map)} entries, recording has {n_channels} channels). "
        "The 16-bit file would not line up with Kilosort's templates."
    )
if ks_positions.shape != rec_positions.shape or not np.allclose(ks_positions, rec_positions):
    raise ValueError(
        "Channel positions differ between the saved recording and Kilosort's "
        "channel_positions.npy. The Kilosort output may come from a different recording."
    )

# ---------- gain (µV per bit) ----------
gains = source.get_channel_gains() if source.has_scaleable_traces() else None
if gains is None:
    gain_to_uV = float(config["bit_to_uv"])
    print(f"[warning] Recording has no gains; using bit_to_uv from config: {gain_to_uV}")
else:
    gains = np.asarray(gains, dtype=float)
    if not np.allclose(gains, gains[0]):
        raise ValueError(f"Channels have different gains ({np.unique(gains)}); MATLAB tools expect one.")
    gain_to_uV = float(gains[0])
    if not np.isclose(gain_to_uV, float(config["bit_to_uv"]), rtol=1e-3):
        print(f"[note] The recording's gain ({gain_to_uV:g} uV/bit, from Open Ephys metadata) is used. "
              f"bit_to_uv_probe_IMEC_calibration_value in 0.2 ({config['bit_to_uv']}) differs; it is only a fallback.")

source_stamp = si_save_path / nam.SETTINGS_STAMP_NAME
int16_settings = {
    "source": str(si_save_path),
    "source_settings": json.loads(source_stamp.read_text()) if source_stamp.exists() else None,
    "n_frames": n_frames,
    "n_channels": n_channels,
    "sampling_rate_hz": fs,
    "dtype": "int16",
    "layout": "interleaved: frame0 ch0..chN-1, frame1 ...",
    "units": "ADC counts, as in the Kilosort input; multiply by gain_to_uV for uV",
    "gain_to_uV": gain_to_uV,
}

# ---------- reuse if it already exists and matches ----------
if dat_path.exists():
    status = nam.check_settings_stamp(dat_path, int16_settings)  # raises if source changed
    if status == "no_stamp":
        raise RuntimeError(
            f"{dat_path} exists but has no settings file, so it can't be checked. "
            "Move or delete it, then rerun this cell."
        )
    if dat_path.stat().st_size != expected_bytes:
        raise ValueError(
            f"{dat_path} has {dat_path.stat().st_size:,} bytes, expected {expected_bytes:,}. "
            "Delete it and rerun this cell."
        )
    print("16-bit file already exists and matches, skipping:", dat_path)
else:
    matlab_folder.mkdir(parents=True, exist_ok=True)

    # ---------- disk space ----------
    free_bytes = shutil.disk_usage(matlab_folder).free
    if free_bytes < expected_bytes * 1.02:
        raise OSError(
            f"Not enough disk space for the 16-bit file: needs {expected_bytes / 1e9:.1f} GB, "
            f"{free_bytes / 1e9:.1f} GB free on that drive."
        )
    print(f"Writing {expected_bytes / 1e9:.2f} GB ({n_channels} ch × {n_frames:,} frames) to:\n  {dat_path}")

    tmp_path = dat_path.with_name(dat_path.name + ".partial")
    chunk_frames = int(fs)  # 1 s per chunk
    int16_max = np.iinfo(np.int16).max
    int16_min = np.iinfo(np.int16).min
    next_report = 0.1
    try:
        with open(tmp_path, "wb") as f:
            for start in range(0, n_frames, chunk_frames):
                end = min(start + chunk_frames, n_frames)
                traces = np.rint(np.asarray(source.get_traces(start_frame=start, end_frame=end)))
                if not np.all(np.isfinite(traces)):
                    raise ValueError(f"Non-finite values in frames {start:,}–{end:,}.")
                lo, hi = traces.min(), traces.max()
                if lo < int16_min or hi > int16_max:
                    raise ValueError(
                        f"Values out of 16-bit range in frames {start:,}–{end:,} "
                        f"(min {lo:g}, max {hi:g}). Nothing was clipped; the file was not written."
                    )
                f.write(traces.astype("<i2").tobytes(order="C"))
                if end / n_frames >= next_report:
                    print(f"  {100 * end / n_frames:.0f}%")
                    next_report += 0.1
    except BaseException:
        tmp_path.unlink(missing_ok=True)
        raise

    if tmp_path.stat().st_size != expected_bytes:
        tmp_path.unlink(missing_ok=True)
        raise OSError("Written file has the wrong size; it was deleted.")

    # ---------- read back the start and end ----------
    written = np.memmap(tmp_path, dtype="<i2", mode="r", shape=(n_frames, n_channels))
    check_frames = min(n_frames, int(fs))
    for a, b in [(0, check_frames), (n_frames - check_frames, n_frames)]:
        expected = np.rint(np.asarray(source.get_traces(start_frame=a, end_frame=b))).astype("<i2")
        if not np.array_equal(np.asarray(written[a:b]), expected):
            del written
            tmp_path.unlink(missing_ok=True)
            raise ValueError(f"Read-back check failed for frames {a:,}–{b:,}; the file was deleted.")
    del written

    tmp_path.replace(dat_path)
    _stamp = nam.write_settings_stamp(dat_path, int16_settings)
    print("16-bit file settings saved:", _stamp.name)
    print("Read-back check passed. 16-bit file saved:", dat_path)

print(f"gain_to_uV = {gain_to_uV}  |  channels = {n_channels}  |  fs = {fs:g} Hz")

### C4 Bombcell and CellExplorer scripts (run in MATLAB)
Writes two ready-to-run MATLAB scripts into `matlab_curation/`, with every path, channel count, sampling rate and gain filled in from this recording:

- **`run_bombcell.m`**: loads the Kilosort output, extracts raw waveforms from the C3 file, computes Bombcell's quality metrics, and writes `cluster_bc_unitType.tsv` to `bombcell/bombcell_<id>/` in the data folder. Kilosort's own folder is left untouched. Bombcell caches raw waveforms and duplicate-spike results in its output folder, so the folder ID changes with the sort **and** the 16-bit input, and the script also sets `reextractRaw` and `recomputeDuplicateSpikes` so nothing cached is reused.
- **`run_cellexplorer.m`**: sets up a CellExplorer session for the C3 file and Kilosort output, loads clusters labelled `good` **or** `mua` (CellExplorer's default is `good` only), and runs `ProcessCellMetrics`. Those are the two labels Kilosort writes, so this covers every Kilosort cluster; any you relabel in Phy (e.g. `noise`) are skipped. The result, `cellexplorer_<fingerprint>.cell_metrics.cellinfo.mat` in that sort's folder, includes the putative cell type of each cluster.

Before writing the scripts, C4 checks that the S2 folder is Kilosort's input (S4 run record, or a Phy curation registered in S5) and that C3's 16-bit file was written from that same S2 folder with the same settings; otherwise it stops.

Then, in MATLAB: open each script and press **Run**. Afterwards, run C5 here.

Each script first checks (SHA-1 of `spike_times.npy` and `spike_clusters.npy`) that Kilosort's output is unchanged since C4 wrote it, and stops if not. When it finishes, it writes a `.settings.json` tag next to its output, which C5 uses to confirm the result belongs to the current sort. CellExplorer runs in its own session folder, `matlab_curation/cellexplorer_<id>/`, because it reuses some cached files (e.g. `*.mono_res.cellinfo.mat`, the monosynaptic connections) even with `forceReload`. The ID changes whenever spike times, cluster assignments, the 16-bit input **or the label files** (`cluster_group.tsv` etc., which decide the units it loads) change, so any change to the set of units starts from an empty folder. `run_cellexplorer.m` also checks the label files and stops if they changed after C4 wrote it.

⚠️ **The scripts are regenerated every time you run this cell**, so they never point at old paths or an old crop. Any edits you make by hand in the `.m` files are overwritten. Change settings here instead.

**Monosynaptic connections** (`CELLEXPLORER_MONOSYNAPTIC`, default `False`): CellExplorer's CCG-based connection detection is slow on long recordings with many units, so it is excluded by default. When switched on, the script also sets `manualAdjustMonoSyn = false`, so CellExplorer doesn't open its manual validation window (its default) and the run can finish unattended.

**Edit:** the three toolbox folders on your machine:
- `BOMBCELL_MATLAB_PATH`: the `matlab` folder inside your Bombcell clone
- `CELLEXPLORER_PATH`: your CellExplorer clone
- `NPY_MATLAB_PATH`: your npy-matlab clone (both tools need it to read `.npy` files)

**Makes:** `run_bombcell.m`, `run_cellexplorer.m` in `matlab_curation/`.

In [ ]:
# ==========================================================================
# [C4] Write the Bombcell and CellExplorer MATLAB scripts
# Needs: C3 (16-bit file), S4 (Kilosort output)
# Makes: <data folder>/matlab_curation/run_bombcell.m, run_cellexplorer.m
# NOTE: both scripts are regenerated on every run; hand edits in the .m files are lost.
# ==========================================================================

import os
import json
import datetime
from pathlib import Path
import Neuropixel_analysis_module as nam

# ----- settings: toolbox folders on this machine -----
BOMBCELL_MATLAB_PATH = r"C:\path\to\bombcell\matlab"   # EDIT: the 'matlab' folder inside your Bombcell clone
CELLEXPLORER_PATH = r"C:\path\to\CellExplorer"          # EDIT
NPY_MATLAB_PATH = r"C:\path\to\npy-matlab"              # EDIT
INT16_FILE_NAME = "spikeband_int16.dat"   # must match C3
# Monosynaptic connections (CCG-based): slow on long recordings with many units. Off by default.
# When on, they are computed without CellExplorer's manual validation window (unattended runs).
CELLEXPLORER_MONOSYNAPTIC = False
# ------------------------------------------------------

config = nam.load_experiment_config(config_path)
nam.define_analysis_paths_from_config(config_path)

sorter_output = Path(nam.kilosort_folder) / "sorter_output"
matlab_folder = Path(nam.data_folder) / "matlab_curation"
dat_path = matlab_folder / INT16_FILE_NAME
dat_settings_path = dat_path.with_name(dat_path.name + ".settings.json")

for path, made_by in [
    (sorter_output / "spike_times.npy", "S4"),
    (dat_path, "C3"),
    (dat_settings_path, "C3"),
]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}\nRun {made_by} first.")

# Inputs must be current: S2 is the folder Kilosort sorted (S4 run record, or a Phy
# curation registered in S5), and the 16-bit file was written from that same S2 folder
si_save_path = nam.preprocessed_folder / config.get("processed_file_name", "si_preprocessed")
nam.verify_sort_input(nam.kilosort_folder, si_save_path)
dat_info = nam.verify_int16_input(dat_path, si_save_path)
print("Inputs OK: S2 = Kilosort's input, and the 16-bit file was written from it.")
n_channels = int(dat_info["n_channels"])
fs = float(dat_info["sampling_rate_hz"])
gain_to_uV = float(dat_info["gain_to_uV"])
# Rechecked by the MATLAB scripts when they run: C3 may be rerun (or the file replaced)
# between C4 and pressing Run in MATLAB
expected_dat_bytes = n_channels * int(dat_info["n_frames"]) * 2
dat_settings_sha1 = nam._file_sha1(dat_settings_path)


# The scripts refuse to run if Kilosort's output changed after C4, and tag their
# results with this fingerprint so C5 can reject results from an older sort
current_sort = nam.sort_fingerprint(sorter_output)

# CellExplorer finds the Kilosort folder relative to its session folder
# One CellExplorer session folder per set of units: CellExplorer reuses cached files such
# as *.mono_res.cellinfo.mat even with forceReload. The name changes with spike times,
# cluster assignments AND the label files (which decide the units it loads).
ce_folder = matlab_folder / nam.cellexplorer_session_name(sorter_output, dat_settings_path)

# Bombcell caches raw waveforms and duplicate-spike results in its output folder:
# one folder per sort AND 16-bit input, and the script forces recomputation as well
bombcell_folder = Path(nam.bombcell_folder) / nam.bombcell_session_name(sorter_output, dat_settings_path)
current_labels = nam.label_fingerprint(sorter_output)
ce_folder.mkdir(parents=True, exist_ok=True)
ks_relative = os.path.relpath(sorter_output, ce_folder)
dat_relative = os.path.relpath(dat_path, ce_folder)


def m_str(value):
    """MATLAB single-quoted string literal."""
    return "'" + str(value).replace("'", "''") + "'"


SEC = "%" * 2   # MATLAB section marker (built here so no notebook line starts with %)
PCT = "%"       # MATLAB comment marker, same reason

# Shared MATLAB pieces: check the sort before running, tag the output after
sort_check = f"""
{SEC} Check this is still the Kilosort run C4 was written for
expectedTimesSha1    = {m_str(current_sort["spike_times_sha1"])};
expectedClustersSha1 = {m_str(current_sort["spike_clusters_sha1"])};
sortFolder = {m_str(sorter_output)};
timesSha1    = sha1File(fullfile(sortFolder, 'spike_times.npy'));
clustersSha1 = sha1File(fullfile(sortFolder, 'spike_clusters.npy'));
assert(strcmp(timesSha1, expectedTimesSha1) && strcmp(clustersSha1, expectedClustersSha1), ...
    'Kilosort output changed since C4 wrote this script. Rerun C4 in the notebook, then this script.');
"""

# Shared: the 16-bit file must still be the one C4 checked (size + its C3 settings file)
dat_check = f"""
{SEC} Check the 16-bit file is still the one C4 checked
datFile = {m_str(dat_path)};
expectedDatBytes = {expected_dat_bytes};
expectedDatSettingsSha1 = {m_str(dat_settings_sha1)};
datListing = dir(datFile);
assert(numel(datListing) == 1 && datListing(1).bytes == expectedDatBytes, ...
    '16-bit file missing or its size changed since C4 (expected %d bytes): %s. Rerun C3 and C4, then this script.', ...
    expectedDatBytes, datFile);
assert(exist([datFile '.settings.json'], 'file') == 2 && ...
    strcmp(sha1File([datFile '.settings.json']), expectedDatSettingsSha1), ...
    '16-bit file settings changed since C4 (C3 rerun?): %s.settings.json. Rerun C4, then this script.', datFile);
"""

# CellExplorer only: the label files decide which units it loads
label_check = "".join(
    (f"assert(exist(fullfile(sortFolder, {m_str(name)}), 'file') == 2 && "
     f"strcmp(sha1File(fullfile(sortFolder, {m_str(name)})), {m_str(h)}), ...\n"
     f"    'Cluster labels ({name}) changed since C4 wrote this script (Phy?). Rerun C4, then this script.');\n")
    if h else
    (f"assert(exist(fullfile(sortFolder, {m_str(name)}), 'file') ~= 2, ...\n"
     f"    '{name} appeared since C4 wrote this script (Phy?). Rerun C4, then this script.');\n")
    for name, h in current_labels.items()
)

sha1_function = f"""
{SEC} Local functions
function h = sha1File(p)
{PCT} SHA-1 of a file's bytes (same as Python hashlib), read in 64 MB chunks
md = java.security.MessageDigest.getInstance('SHA-1');
fid = fopen(p, 'r');
assert(fid > 0, 'Cannot open %s', p);
cleaner = onCleanup(@() fclose(fid));
while true
    d = fread(fid, 64 * 1024 * 1024, '*uint8');
    if isempty(d), break; end
    md.update(typecast(d, 'int8'));
end
h = lower(reshape(dec2hex(typecast(md.digest(), 'uint8'), 2)', 1, []));
end

function writeSortTag(outFile, timesSha1, clustersSha1)
{PCT} Sidecar read by notebook cell C5: ties outFile to this exact Kilosort run
fid = fopen([outFile '.settings.json'], 'w');
assert(fid > 0, 'Cannot write %s.settings.json', outFile);
fprintf(fid, '{{"spike_clusters_sha1": "%s", "spike_times_sha1": "%s"}}\\n', clustersSha1, timesSha1);
fclose(fid);
end
"""
stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
header = (
    f"% Auto-generated by notebook cell C4 on {stamp}.\n"
    "% Regenerated every time C4 runs: edit the notebook settings, not this file.\n"
)

bombcell_script = header + f"""
{SEC} Paths
bombcellPath      = {m_str(BOMBCELL_MATLAB_PATH)};
npyMatlabPath     = {m_str(NPY_MATLAB_PATH)};
ephysKilosortPath = {m_str(sorter_output)};
ephysRawFile      = {m_str(dat_path)};
savePath          = {m_str(bombcell_folder)};

assert(exist(bombcellPath, 'dir') == 7, 'Bombcell folder not found: %s (set BOMBCELL_MATLAB_PATH in C4)', bombcellPath);
assert(exist(npyMatlabPath, 'dir') == 7, 'npy-matlab folder not found: %s (set NPY_MATLAB_PATH in C4)', npyMatlabPath);
assert(exist(ephysRawFile, 'file') == 2, '16-bit file not found: %s (run C3)', ephysRawFile);
addpath(genpath(bombcellPath));
addpath(genpath(npyMatlabPath));
if ~exist(savePath, 'dir'), mkdir(savePath); end
{sort_check}{dat_check}
bcOut = fullfile(savePath, 'cluster_bc_unitType.tsv');
if exist([bcOut '.settings.json'], 'file'), delete([bcOut '.settings.json']); end  % no valid tag until this run finishes

{SEC} Recording facts (from the C3 settings file)
nChannels       = {n_channels};
sampleRate      = {fs!r};
gain_to_uV      = {gain_to_uV!r};   % uV per bit of the 16-bit file
kilosortVersion = 4;

{SEC} Load Kilosort output
[spikeTimes_samples, spikeClusters, templateWaveforms, templateAmplitudes, ...
    pcFeatures, pcFeatureIdx, channelPositions] = bc.load.loadEphysData(ephysKilosortPath, savePath);

{SEC} Parameters
param = bc.qm.qualityParamValues([], ephysRawFile, ephysKilosortPath, gain_to_uV, kilosortVersion);
param.nChannels         = nChannels;   % the 16-bit file has no sync channel
param.nSyncChannels     = 0;
param.ephys_sample_rate = sampleRate;
param.spikeWidth        = bc.qm.helpers.calculateSpikeWidth(sampleRate, kilosortVersion);
param.ephysKilosortPath = savePath;    % write cluster_bc_unitType.tsv here, not into Kilosort's folder
param.reextractRaw              = 1;   % never reuse cached raw waveforms
param.recomputeDuplicateSpikes  = 1;   % never reuse cached duplicate-spike results

{SEC} Run
[qMetric, unitType] = bc.qm.runAllQualityMetrics(param, spikeTimes_samples, spikeClusters, ...
    templateWaveforms, templateAmplitudes, pcFeatures, pcFeatureIdx, channelPositions, savePath);

assert(exist(bcOut, 'file') == 2, 'Bombcell did not write %s (check param.saveAsTSV / unitType_for_phy)', bcOut);
writeSortTag(bcOut, timesSha1, clustersSha1);
fprintf('\\nBombcell finished. Unit types: %s\\n', bcOut);
{sha1_function}"""

# Monosynaptic connections: excluded unless CELLEXPLORER_MONOSYNAPTIC (MATLAB cell array of metric names)
mono_exclude = "" if CELLEXPLORER_MONOSYNAPTIC else "'excludeMetrics', {'monoSynaptic_connections'}, "

cellexplorer_script = header + f"""
{SEC} Paths
cellExplorerPath = {m_str(CELLEXPLORER_PATH)};
npyMatlabPath    = {m_str(NPY_MATLAB_PATH)};
basepath         = {m_str(ce_folder)};   % one folder per sort (no stale caches)
datFileName      = {m_str(dat_relative)};
ksRelativePath   = {m_str(ks_relative)};

assert(exist(cellExplorerPath, 'dir') == 7, 'CellExplorer folder not found: %s (set CELLEXPLORER_PATH in C4)', cellExplorerPath);
assert(exist(npyMatlabPath, 'dir') == 7, 'npy-matlab folder not found: %s (set NPY_MATLAB_PATH in C4)', npyMatlabPath);
assert(exist(fullfile(basepath, datFileName), 'file') == 2, '16-bit file not found (run C3)');
assert(exist(fullfile(basepath, ksRelativePath, 'spike_times.npy'), 'file') == 2, 'Kilosort output not found');
addpath(genpath(cellExplorerPath));
addpath(genpath(npyMatlabPath));
{sort_check}{label_check}{dat_check}
{SEC} Session
session = sessionTemplate(basepath, 'showGUI', false);
session.extracellular.sr                  = {fs!r};
session.extracellular.nChannels           = {n_channels};
session.extracellular.precision           = 'int16';
session.extracellular.leastSignificantBit = {gain_to_uV!r};   % uV per bit
session.extracellular.fileName            = datFileName;
session.extracellular.electrodeGroups.channels = {{1:{n_channels}}};
session.extracellular.nElectrodeGroups    = 1;
session.extracellular.spikeGroups         = session.extracellular.electrodeGroups;
session.extracellular.nSpikeGroups        = 1;

pos = double(readNPY(fullfile(basepath, ksRelativePath, 'channel_positions.npy')));
session.extracellular.chanCoords.x      = pos(:, 1)';
session.extracellular.chanCoords.y      = pos(:, 2)';
session.extracellular.chanCoords.source = 'Kilosort channel_positions.npy';
session.extracellular.chanCoords.layout = 'Unknown';
yu = unique(pos(:, 2));
session.extracellular.chanCoords.verticalSpacing = median(diff(yu));

session.spikeSorting = {{}};
session.spikeSorting{{1}}.relativePath    = ksRelativePath;
session.spikeSorting{{1}}.format          = 'Phy';
session.spikeSorting{{1}}.method          = 'KiloSort4';
session.spikeSorting{{1}}.channels        = [];
session.spikeSorting{{1}}.manuallyCurated = 0;
session.spikeSorting{{1}}.notes           = 'Auto-configured by notebook cell C4';

{SEC} Load clusters labelled good or mua: all of Kilosort's own labels; clusters relabelled
{PCT} in Phy as noise (or other labels) are skipped. CellExplorer's default reads only 'good'.
spikes = loadSpikes('session', session, 'labelsToRead', {{'good', 'mua'}}, ...
    'getWaveformsFromDat', true, 'forceReload', true, 'showWaveforms', false);

{SEC} Cell metrics and putative cell types
cell_metrics = ProcessCellMetrics('session', session, 'spikes', spikes, ...
    'showGUI', false, 'showWaveforms', false, 'sessionSummaryFigure', false, ...
    'manualAdjustMonoSyn', false, {mono_exclude}...
    'forceReload', true);   % recompute everything; never reuse metrics from an older sort

ceOut = fullfile(basepath, [session.general.name '.cell_metrics.cellinfo.mat']);
assert(exist(ceOut, 'file') == 2, 'CellExplorer did not write %s', ceOut);
writeSortTag(ceOut, timesSha1, clustersSha1);
fprintf('\\nCellExplorer finished: %d units. Output in %s\\n', numel(cell_metrics.cluID), basepath);
{sha1_function}"""

matlab_folder.mkdir(parents=True, exist_ok=True)
bombcell_script_path = matlab_folder / "run_bombcell.m"
cellexplorer_script_path = matlab_folder / "run_cellexplorer.m"
bombcell_script_path.write_text(bombcell_script)
cellexplorer_script_path.write_text(cellexplorer_script)

for label, folder in [
    ("BOMBCELL_MATLAB_PATH", BOMBCELL_MATLAB_PATH),
    ("CELLEXPLORER_PATH", CELLEXPLORER_PATH),
    ("NPY_MATLAB_PATH", NPY_MATLAB_PATH),
]:
    if not Path(folder).is_dir():
        print(f"[warning] {label} not found on this machine: {folder}")

print("Scripts written (overwritten if they existed):")
print("  ", bombcell_script_path)
print("  ", cellexplorer_script_path)
print("CellExplorer monosynaptic connections:", "ON (no manual validation window)" if CELLEXPLORER_MONOSYNAPTIC else "OFF")
print("Open them in MATLAB and press Run, then run C5.")

### C5 Unit summary table
One row per Kilosort cluster, merging whatever results exist:

| Source | Columns | Counts as good when |
|---|---|---|
| Kilosort / Phy (always) | `ks_label` (automatic), `phy_label` (from `cluster_group.tsv`), `n_spikes`, `firing_rate_hz` | `phy_label` is `good` where it exists (your manual Phy label wins), else `ks_label` is `good` |
| UnitRefine (C2) | `ur_noise_label`, `ur_sua_mua_label`, probabilities | `neural` **and** `sua` (a missing or unexpected label is never good) |
| Bombcell (C4, MATLAB) | `bc_unit_type` | `GOOD` |
| CellExplorer (C4, MATLAB) | `ce_putative_cell_type` | (information only, no quality vote) |

`consensus_good` is True only when **every tool that was run** calls the unit good **and** you didn't reject it by hand in Phy (`manually_rejected`: relabelled `noise` or `mua`). Kilosort's automatic label and your Phy label are kept in separate columns. Merging or splitting clusters in Phy changes `spike_clusters.npy`: run **S5** to register the curated version, then rerun C1–C5 (the fingerprint checks enforce this). `n_quality_sources` says how many tools voted, and `n_sources_good` how many said good. Firing rate uses the full cropped recording length, taken from the S2 folder after checking it is the one Kilosort sorted.

Every curation result must carry the fingerprint of the **current** Kilosort output (written by C2 and by the MATLAB scripts). This includes a Bombcell file sitting inside `sorter_output`: its location alone doesn't prove which sort it came from. A result from an older sort, or one without a tag, stops the cell with what to rerun; matching cluster IDs alone are not accepted. UnitRefine counts as good only when both labels are present and valid (`neural` and `sua`). An old UnitRefine file with made-up sequential IDs is ignored with a warning; rerun C2 to replace it. You can rerun C5 any time, e.g. after Bombcell finishes in MATLAB.

Bombcell and CellExplorer results are read from the folders named after the current sort and 16-bit file (the same names C4 uses); if C3's file is from other preprocessing, the cell stops.

**Edit:** `INT16_FILE_NAME` (must match C3 / C4).

**Makes:** `unit_summary`, plus `unit_summary.csv` and `consensus_good_units.csv` in `custom_analytics/`.

In [ ]:
# ==========================================================================
# [C5] Unit summary: Kilosort + whatever curation results exist
# Needs: S4 (Kilosort output); uses C2 / Bombcell / CellExplorer results if present,
#        only if they are tagged with the current sort's fingerprint
# Makes: unit_summary, custom_analytics/unit_summary.csv, consensus_good_units.csv
# ==========================================================================

import json
import pandas as pd
from pathlib import Path
import Neuropixel_analysis_module as nam

# ----- settings -----
INT16_FILE_NAME = "spikeband_int16.dat"   # must match C3 / C4
# --------------------

config = nam.load_experiment_config(config_path)
nam.define_analysis_paths_from_config(config_path)

sorter_output = Path(nam.kilosort_folder) / "sorter_output"
matlab_folder = Path(nam.data_folder) / "matlab_curation"
custom_analytics = Path(nam.custom_analytics_folder)
custom_analytics.mkdir(parents=True, exist_ok=True)

if not (sorter_output / "spike_clusters.npy").exists():
    raise FileNotFoundError(f"Missing Kilosort output in {sorter_output}\nRun S4 first.")

# Recording length for firing rates: from the S2 folder Kilosort actually sorted
si_save_path = nam.preprocessed_folder / config.get("processed_file_name", "si_preprocessed")
nam.verify_sort_input(nam.kilosort_folder, si_save_path)
stamp = json.loads((si_save_path / nam.SETTINGS_STAMP_NAME).read_text())
fs, n_frames = float(stamp["raw_sampling_rate_hz"]), int(stamp["raw_n_frames"])

ks_df = nam.read_kilosort_units(sorter_output, fs, n_frames=n_frames)
ur_df = nam.read_unit_refine(nam.unit_refine_folder)
# MATLAB results live in folders named after the sort AND the 16-bit input (same names as C4).
# Without a current 16-bit file there can't be current MATLAB results.
dat_path = matlab_folder / INT16_FILE_NAME
dat_settings_path = dat_path.with_name(dat_path.name + ".settings.json")
bc_folder = ce_folder = None
if dat_settings_path.exists():
    nam.verify_int16_input(dat_path, si_save_path)   # stops if C3's file is from other preprocessing
    bc_folder = Path(nam.bombcell_folder) / nam.bombcell_session_name(sorter_output, dat_settings_path)
    ce_folder = matlab_folder / nam.cellexplorer_session_name(sorter_output, dat_settings_path)
bc_df = nam.read_bombcell(bc_folder, sorter_output) if bc_folder is not None else nam.read_bombcell(sorter_output)
ce_df = nam.read_cellexplorer(ce_folder) if ce_folder is not None and ce_folder.exists() else None

# ---------- every curation result must come from THIS Kilosort run ----------
# Matching cluster IDs are not enough: a rerun of Kilosort can reuse the same IDs.
current_sort = nam.sort_fingerprint(sorter_output)


def _require_same_sort(name, result_path, rerun_hint):
    try:
        status = nam.check_settings_stamp(result_path, current_sort)
    except ValueError:
        raise ValueError(
            f"{name} results come from a different Kilosort run than the current sorter_output:\n"
            f"{result_path}\n{rerun_hint}"
        ) from None
    if status == "no_stamp":
        raise ValueError(
            f"{name} results have no sort tag, so they can't be matched to this Kilosort run:\n"
            f"{result_path}\n{rerun_hint}"
        )


if ur_df is not None:
    _require_same_sort("UnitRefine", Path(nam.unit_refine_folder) / "unit_refine_results.csv",
                       "Rerun C2 (and C1 if it stops), or delete the file to leave UnitRefine out.")
if bc_df is not None:
    # Same folder order as read_bombcell; a copy inside sorter_output needs a tag too
    bc_path = next(p for p in ([bc_folder / "cluster_bc_unitType.tsv"] if bc_folder is not None else [])
                   + [sorter_output / "cluster_bc_unitType.tsv"] if p.exists())
    _require_same_sort("Bombcell", bc_path,
                       "Rerun C4, then run_bombcell.m in MATLAB, or delete the file to leave Bombcell out.")
if ce_df is not None:
    ce_path = sorted(ce_folder.glob("*.cell_metrics.cellinfo.mat"), key=lambda p: p.stat().st_mtime)[-1]
    _require_same_sort("CellExplorer", ce_path,
                       "Rerun C4, then run_cellexplorer.m in MATLAB, or delete the file to leave CellExplorer out.")

# Cluster IDs in the curation results must be Kilosort's
ks_ids = set(ks_df["cluster_id"])
for name, df in [("UnitRefine", ur_df), ("Bombcell", bc_df), ("CellExplorer", ce_df)]:
    if df is None:
        continue
    unknown = sorted(set(df["cluster_id"]) - ks_ids)
    if unknown:
        raise ValueError(
            f"{name} has {len(unknown)} cluster IDs that are not in this Kilosort output "
            f"(e.g. {unknown[:5]}). It was probably run on an older sort; rerun it."
        )
    missing = len(ks_ids - set(df["cluster_id"]))
    if missing:
        print(f"Note: {name} has no row for {missing} of {len(ks_ids)} Kilosort clusters "
              "(shown as empty / not good).")

unit_summary, sources_used = nam.build_unit_summary(ks_df, ur_df, bc_df, ce_df)

summary_path = custom_analytics / "unit_summary.csv"
good_path = custom_analytics / "consensus_good_units.csv"
try:
    unit_summary.to_csv(summary_path, index=False)
    unit_summary.loc[unit_summary["consensus_good"], "cluster_id"].to_csv(good_path, index=False)
except PermissionError:
    raise PermissionError(f"Can't write {summary_path}. Close it in Excel and rerun.")

print("Sources used:", ", ".join(sources_used))
print(f"Kilosort clusters: {len(unit_summary)}")
print(f"Labels edited by hand in Phy: {int(unit_summary['phy_label_edited'].sum())}  |  "
      f"manually rejected (noise/mua in Phy): {int(unit_summary['manually_rejected'].sum())}")
for col in [c for c in unit_summary.columns if c.startswith("good_")]:
    print(f"  {col[5:]:<12} good: {int(unit_summary[col].sum())}")
n_quality = sum(c.startswith("good_") for c in unit_summary.columns)
print(f"Consensus good (all {n_quality} quality sources agree): "
      f"{int(unit_summary['consensus_good'].sum())}")
if "ce_putative_cell_type" in unit_summary.columns:
    print("\nCellExplorer putative types among consensus-good units:")
    print(unit_summary.loc[unit_summary["consensus_good"], "ce_putative_cell_type"]
          .value_counts(dropna=False).to_string())
print("\nSaved:", summary_path)
print("Saved:", good_path)

---
## 4 · LFP branch
Independent of the spike branch. Requires 1.1 only.

### L1 LFP filter and downsample
Bandpass 1–300 Hz, then resample to 1250 Hz. Produces two versions:
- **No reference**: best for biological LFP, oscillations, waves.
- **Global median CMR**: useful for spotting common-mode artifacts by comparison.

The filter uses a 3 s chunk overlap (`margin_ms`). The 1 Hz high-pass needs seconds of surrounding signal to settle; SpikeInterface's 5 ms default would leave small artifacts at every chunk boundary.

**Edit:** `LFP_FREQ_MIN`, `LFP_FREQ_MAX`, `LFP_RESAMPLE_HZ`.

**Makes:** `rec_step7_lfp_no_ref`, `rec_step7_lfp_cmr`, `lfp_settings`.

In [ ]:
# ==========================================================================
# [L1] LFP filter (1–300 Hz) and resample to 1250 Hz
# Needs: 1.1 (rec_step4)
# Makes: rec_step7_lfp_no_ref, rec_step7_lfp_cmr, lfp_settings
# ==========================================================================

import spikeinterface.full as si

# ----- settings -----
LFP_FREQ_MIN = 1.0
LFP_FREQ_MAX = 300.0
LFP_RESAMPLE_HZ = 1250
LFP_FILTER_MARGIN_MS = 3000.0   # chunk overlap for the 1 Hz high-pass (default 5 ms is too short)
# --------------------

rec_step5_lfp = si.bandpass_filter(
    rec_step4,
    freq_min=LFP_FREQ_MIN,
    freq_max=LFP_FREQ_MAX,
    margin_ms=LFP_FILTER_MARGIN_MS,
    dtype="float32",
)

# No-reference LFP: biological LFP / oscillations / waves
rec_step7_lfp_no_ref = si.resample(rec_step5_lfp, resample_rate=LFP_RESAMPLE_HZ)

# CMR LFP: artifact / common-mode comparison
rec_step6_lfp_cmr = si.common_reference(rec_step5_lfp, operator="median", reference="global")
rec_step7_lfp_cmr = si.resample(rec_step6_lfp_cmr, resample_rate=LFP_RESAMPLE_HZ)

lfp_settings = {
    **shared_settings,
    "lfp_freq_min": LFP_FREQ_MIN,
    "lfp_freq_max": LFP_FREQ_MAX,
    "lfp_filter_margin_ms": LFP_FILTER_MARGIN_MS,
    "lfp_resample_hz": LFP_RESAMPLE_HZ,
    "dtype": "float32",
}

print("No-ref LFP:", rec_step7_lfp_no_ref)
print("CMR LFP:   ", rec_step7_lfp_cmr)
print(f"LFP duration: {rec_step7_lfp_no_ref.get_total_duration() / 60:.2f} min")

### L2 Save LFP recordings
Saves the no-ref and CMR versions to separate folders. Same safety check as S2: each folder gets a settings file, and an existing folder is reused only if its length and saved settings match. A folder without a settings file stops the cell (move or delete it and rerun).

**Edit:** `SAVE_LFP_NOREF` / `SAVE_LFP_CMR` to skip either version; `LFP_SAVE_CHUNK_DURATION` (default `"10s"`) and `LFP_SAVE_N_JOBS` (default 4) set memory use. Each worker filters its chunk plus 3 s on each side at 30 kHz on all channels (about 1.4 GB of float32 per 30 s for 384 channels, more with filter temporaries); the cell prints an estimate before saving. Lower the workers or the chunk if RAM is short. These don't change the result.

**Makes:** `si_preprocessed_lfp_noref/` and `si_preprocessed_lfp_cmr/` in the preprocessed folder.

In [ ]:
# ==========================================================================
# [L2] Save LFP recordings
# Needs: L1
# Makes: <preprocessed>/si_preprocessed_lfp_noref/, <preprocessed>/si_preprocessed_lfp_cmr/
# ==========================================================================

import Neuropixel_analysis_module as nam
import spikeinterface.full as si

# ----- settings -----
SAVE_LFP_NOREF = True
SAVE_LFP_CMR = True
# Each worker filters (chunk + 2 x 3 s margin) of 30 kHz data for all channels, so RAM grows with
# both settings. The estimate below is printed before saving; lower these if RAM is short.
LFP_SAVE_CHUNK_DURATION = "10s"
LFP_SAVE_N_JOBS = 4
# --------------------

config = nam.load_experiment_config(config_path)
nam.define_analysis_paths_from_config(config_path)

output_folder = nam.preprocessed_folder

base_name = config.get("processed_lfp_file_name", "si_preprocessed_lfp")

if "lfp_settings" not in globals():
    raise RuntimeError("Run L1 first (lfp_settings missing).")
output_folder.mkdir(parents=True, exist_ok=True)

lfp_versions = [
    (f"{base_name}_noref", rec_step7_lfp_no_ref, SAVE_LFP_NOREF, {**lfp_settings, "reference": "none"}),
    (f"{base_name}_cmr",   rec_step7_lfp_cmr,    SAVE_LFP_CMR,   {**lfp_settings, "reference": "global median"}),
]

_chunk_s = float(str(LFP_SAVE_CHUNK_DURATION).rstrip("s"))
_raw_bytes_per_s = rec_step4.get_num_channels() * rec_step4.get_sampling_frequency() * 4
_per_worker_gb = (_chunk_s + 2 * lfp_settings["lfp_filter_margin_ms"] / 1000) * _raw_bytes_per_s * 3 / 1e9
print(f"LFP save: chunks of {LFP_SAVE_CHUNK_DURATION}, {LFP_SAVE_N_JOBS} workers, "
      f"roughly {_per_worker_gb:.1f} GB RAM per worker ({_per_worker_gb * max(LFP_SAVE_N_JOBS, 1):.0f} GB total, estimate)")

for folder_name, lfp_rec, do_save, folder_settings in lfp_versions:
    if not do_save:
        print(f"Skipping (disabled): {folder_name}")
        continue

    save_path = output_folder / folder_name

    if save_path.exists():
        existing = si.load(save_path)
        if existing.get_num_frames() != lfp_rec.get_num_frames():
            raise ValueError(
                f"Existing LFP folder does not match the current crop: {save_path}\n"
                f"saved frames = {existing.get_num_frames():,}, "
                f"current frames = {lfp_rec.get_num_frames():,}. Move or delete it."
            )
        status = nam.check_settings_stamp(save_path, folder_settings)
        if status == "no_stamp":
            # Never stamped after the fact: settings of an existing folder can't be known
            raise RuntimeError(
                "This folder has no settings file, so what it contains can't be checked:\n"
                f"{save_path}\n"
                "It was saved by an older notebook version, or the save didn't finish. Move or delete it, "
                "then rerun L2 to write a new one."
            )
        print(f"Folder already exists and matches current settings, skipping: {save_path}")
        continue

    lfp_rec.save(
        folder=save_path,
        chunk_duration=LFP_SAVE_CHUNK_DURATION,
        n_jobs=LFP_SAVE_N_JOBS,
        progress_bar=True,
    )
    _stamp = nam.write_settings_stamp(save_path, folder_settings)
    print(f"LFP saved to: {save_path} (settings: {_stamp.name})")

---
## 5 · Spike–LFP alignment (BETA)
Run after **S3, S4 and L2**. Puts the Kilosort spikes and the LFP on one shared clock so they can be analysed together.

### A1 Spike–LFP alignment files (BETA)

> ⚠️ **BETA: not yet tested on real data.** Optional, and only needed for spike–LFP analysis. Nothing above depends on it. Before relying on it, check that the printed spike and LFP time ranges look right; if you need the LFP features, ask to have this cell tested first.

The spikes (30 kHz) and the LFP (1250 Hz) come from the same cropped recording, so they share sample 0. This cell writes small lookup files that put both onto the **original Open Ephys clock**, the same clock as `timestamps.npy` and your event files.

Before aligning, it checks provenance, not just lengths: the spike folder (S2), LFP folder (L2) and trimmed timestamps (S3) must share the same crop, channels and source recording, and Kilosort's run record must name the current spike folder, with the same settings, and match the current `sorter_output` fingerprint. The trimmed timestamps must have been **read** from Open Ephys files by S3: outputs of older notebook versions that inferred sample numbers, or that carry no provenance, are refused (not deleted; rerun S3). Timestamps stored as sample counts are converted to seconds. LFP samples that fall past the last raw frame (rounding at the very end) get timestamp NaN and `lfp_valid` False; nothing is extrapolated.

**Discontinuities:** repeated or backward samples stop the cell, because the clock can't be trusted. Forward gaps (lost samples) don't stop it, but the recording is joined across each gap and every filter rings there, so LFP samples within 3 cycles of the LFP high-pass (3 s at 1 Hz) of a gap are marked invalid in `lfp_valid.npy`, and spikes falling there in `spike_near_gap.npy`. Drop both before phase analysis.

Everything is loaded from disk, so it also works after a kernel restart. Kilosort's own files are not modified.

**Writes to `<preprocessed>/alignment/`:**

| File | One value per | Meaning |
|---|---|---|
| `lfp_timestamps.npy` | LFP sample | Open Ephys time (s) of that LFP sample |
| `spike_times_oe_s.npy` | spike (same order as Kilosort's `spike_times.npy` / `spike_clusters.npy`) | Open Ephys time (s) of that spike |
| `spike_lfp_index.npy` | spike | Nearest LFP sample to that spike |
| `lfp_valid.npy` | LFP sample | False near a forward gap (don't use for phase) |
| `spike_near_gap.npy` | spike | True if the spike's LFP sample is near a forward gap |
| `alignment_info.json` | — | Sampling rates, ratio, crop origin, frame counts, gaps |

**Typical use:**
```python
lfp = si.load(lfp_folder).get_traces(channel_ids=[ch])[:, 0]
idx = np.load(".../alignment/spike_lfp_index.npy")
clu = np.load(".../sorter_output/spike_clusters.npy")
ok = ~np.load(".../alignment/spike_near_gap.npy")
lfp_at_unit5_spikes = lfp[idx[(clu == 5) & ok]]
```
For phase locking, filter the LFP to your band, take the Hilbert phase, and index it with `spike_lfp_index` (skipping spikes near gaps). For sub-sample precision, **don't interpolate the phase angle directly**: it wraps at ±π, so a straight line between +3.1 and −3.1 rad passes through 0 instead of ±π. Interpolate the analytic signal's real and imaginary parts (or cos and sin of the phase) at `spike_times_oe_s` using `lfp_timestamps`, then take the angle:
```python
z = scipy.signal.hilbert(lfp_band)
re = np.interp(spike_t, lfp_t, z.real); im = np.interp(spike_t, lfp_t, z.imag)
spike_phase = np.angle(re + 1j * im)
```

**Phase timing:** SpikeInterface filters run forwards and backwards by default (zero phase), so the LFP has no filter delay relative to the spikes. Keep that in mind if you filter the LFP further yourself: use `filtfilt`/`sosfiltfilt`, not a one-way filter.

**Edit:** nothing.

In [ ]:
# ==========================================================================
# [A1] Spike–LFP alignment files (shared Open Ephys clock)  —  BETA, NOT YET TESTED
# Needs: S3 (timestamps_trimmed.npy), S4 (Kilosort output), L2 (saved no-ref LFP)
# Makes: <preprocessed>/alignment/ lfp_timestamps.npy, spike_times_oe_s.npy,
#        spike_lfp_index.npy, lfp_valid.npy, spike_near_gap.npy, alignment_info.json
# ==========================================================================

import json
import numpy as np
from pathlib import Path
import spikeinterface.full as si
import Neuropixel_analysis_module as nam

config = nam.load_experiment_config(config_path)
nam.define_analysis_paths_from_config(config_path)

# ============================================================
# LOAD INPUTS FROM DISK
# ============================================================

spike_rec_path = nam.preprocessed_folder / config.get("processed_file_name", "si_preprocessed")
lfp_rec_path = nam.preprocessed_folder / f'{config.get("processed_lfp_file_name", "si_preprocessed_lfp")}_noref'
timestamps_path = nam.preprocessed_folder / "timestamps_trimmed.npy"
ks_out = Path(nam.kilosort_folder) / "sorter_output"
ks_spike_times_path = ks_out / "spike_times.npy"

for path, made_by in [
    (spike_rec_path, "S2"),
    (lfp_rec_path, "L2"),
    (timestamps_path, "S3"),
    (ks_spike_times_path, "S4"),
]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}\nRun {made_by} first.")

spike_rec = si.load(spike_rec_path)
lfp_rec = si.load(lfp_rec_path)

fs_raw = float(spike_rec.get_sampling_frequency())
fs_lfp = float(lfp_rec.get_sampling_frequency())
n_raw = int(spike_rec.get_num_frames())
n_lfp = int(lfp_rec.get_num_frames())
raw_per_lfp = fs_raw / fs_lfp

# Older Open Ephys versions store sample counts in timestamps.npy: convert to seconds
ts_clock_report = {}
timestamps_trimmed, timestamps_were_counts = nam.timestamps_to_seconds(
    np.load(timestamps_path, mmap_mode="r"), fs_raw, report=ts_clock_report
)
if ts_clock_report["n_backward_corrections"]:
    print(f"Timestamps: {ts_clock_report['n_backward_corrections']} Open Ephys synchronization correction(s), "
          f"backward steps up to {1e6 * ts_clock_report['largest_backward_correction_s']:.1f} us (accepted).")
if timestamps_were_counts:
    print("Note: timestamps.npy holds sample counts; converted to seconds (count / fs).")

# ============================================================
# DISCONTINUITIES
# Repeats / backward steps: the clock can't be trusted -> stop (timestamps_to_seconds
# also stops on them). Forward gaps (lost samples): the recording is joined across
# the gap and every filter rings there, so LFP samples near a gap are masked.
# ============================================================

sample_numbers_path = nam.preprocessed_folder / "sample_numbers_trimmed.npy"
if not sample_numbers_path.exists():
    raise FileNotFoundError(f"Missing {sample_numbers_path}\nRun S3 first.")
d_sn = np.diff(np.load(sample_numbers_path, mmap_mode="r").astype(np.int64))
if np.any(d_sn <= 0):
    raise ValueError(
        f"sample_numbers_trimmed.npy has {int(np.count_nonzero(d_sn == 0))} repeated and "
        f"{int(np.count_nonzero(d_sn < 0))} backward steps. Alignment would be wrong; "
        "see timestamp_gap_report.json (S3)."
    )
forward_gaps = np.flatnonzero(d_sn > 1)          # gap between raw frame i and i+1
# validated before the integer conversion (NaN/inf, fractional, negative -> error)
spike_samples = nam.load_spike_samples(ks_spike_times_path)

print(f"Raw:  {fs_raw:g} Hz, {n_raw:,} frames")
print(f"LFP:  {fs_lfp:g} Hz, {n_lfp:,} frames  ({raw_per_lfp:g} raw samples per LFP sample)")
print(f"Spikes: {len(spike_samples):,}")

# ============================================================
# PROVENANCE CHECKS: same crop, channels and Kilosort run everywhere
# (lengths alone can match by coincidence)
# ============================================================

def _read_stamp(path, made_by):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"Missing settings file {path}\nRerun {made_by} with this notebook version so A1 can check it."
        )
    return json.loads(Path(path).read_text())


spike_stamp = _read_stamp(spike_rec_path / nam.SETTINGS_STAMP_NAME, "S2")
lfp_stamp = _read_stamp(lfp_rec_path / nam.SETTINGS_STAMP_NAME, "L2")
# S3 outputs must be READ from Open Ephys files (inferred or provenance-less ones are refused)
# and match the S2 folder; raises with what to rerun
ts_stamp, ts_gap_report = nam.verify_trimmed_timestamps(nam.preprocessed_folder, spike_rec_path)

shared_keys = list(nam.SHARED_SETTINGS_KEYS)   # settings shared by both branches
for label, stamp in [("spike folder (S2)", spike_stamp), ("LFP folder (L2)", lfp_stamp)]:
    diffs = [k for k in shared_keys if stamp.get(k) != ts_stamp.get(k)]
    if diffs:
        raise ValueError(
            f"The {label} and the trimmed timestamps (S3) come from different preprocessing:\n"
            + "\n".join(f"  {k}: {label} = {stamp.get(k)!r}, timestamps = {ts_stamp.get(k)!r}" for k in diffs)
            + "\nRerun the stale step(s) so all three use the same crop and channels."
        )

# S2 folder = Kilosort's input, and sorter_output unchanged since S4
nam.verify_sort_input(nam.kilosort_folder, spike_rec_path)
print("Provenance checks passed: spikes, LFP, timestamps and Kilosort share one crop and channel set.")

# ============================================================
# CONSISTENCY CHECKS
# ============================================================

if len(timestamps_trimmed) != n_raw:
    raise ValueError(
        f"timestamps_trimmed.npy has {len(timestamps_trimmed):,} frames but the spike "
        f"recording has {n_raw:,}. Rerun S3."
    )

expected_n_lfp = n_raw / raw_per_lfp
if abs(n_lfp - expected_n_lfp) > 2:
    raise ValueError(
        f"LFP length ({n_lfp:,}) does not match the spike recording "
        f"(expected about {expected_n_lfp:,.0f}). They were probably saved with "
        "different crops. Re-save the stale one."
    )

if spike_samples.size and (spike_samples.min() < 0 or spike_samples.max() >= n_raw):
    raise ValueError(
        "Kilosort spike samples fall outside the spike recording. "
        "The Kilosort output may come from a different recording/crop."
    )

# ============================================================
# LFP SAMPLE -> OPEN EPHYS TIME
# LFP sample k sits at raw position k * raw_per_lfp (SpikeInterface resampling keeps sample 0 aligned).
# Linear interpolation handles non-integer ratios; for 30000/1250 it is an exact lookup.
# ============================================================

raw_pos = np.arange(n_lfp, dtype=np.float64) * raw_per_lfp
i0 = np.floor(raw_pos).astype(np.int64)
n_past_end = int(np.count_nonzero(i0 > n_raw - 1))
i0 = np.clip(i0, 0, n_raw - 1)
i1 = np.clip(i0 + 1, 0, n_raw - 1)
frac = raw_pos - np.floor(raw_pos)

ts0 = np.asarray(timestamps_trimmed[i0], dtype=np.float64)
ts1 = np.asarray(timestamps_trimmed[i1], dtype=np.float64)
lfp_timestamps = ts0 + frac * (ts1 - ts0)

if n_past_end:
    # Last LFP samples (rounding at the very end) fall past the last raw frame: no recorded time
    # exists for them, so they get NaN and are marked invalid below (nothing is extrapolated)
    lfp_timestamps[n_lfp - n_past_end:] = np.nan
    print(f"Note: last {n_past_end} LFP sample(s) fall past the final raw frame: timestamp NaN, lfp_valid False.")

# ============================================================
# SPIKES -> OPEN EPHYS TIME AND NEAREST LFP SAMPLE
# ============================================================

spike_times_oe_s = np.asarray(timestamps_trimmed[spike_samples], dtype=np.float64)
spike_lfp_index = np.clip(
    np.rint(spike_samples / raw_per_lfp).astype(np.int64), 0, n_lfp - 1
)

# ============================================================
# MASK AROUND FORWARD GAPS
# Margin = 3 cycles of the LFP high-pass (3 s at 1 Hz), so filter ringing has settled.
# ============================================================

lfp_freq_min = float(lfp_stamp.get("lfp_freq_min") or 1.0)
gap_margin_s = 3.0 / lfp_freq_min
lfp_valid = np.ones(n_lfp, dtype=bool)
if n_past_end:
    lfp_valid[n_lfp - n_past_end:] = False
gap_lfp_positions = (forward_gaps + 0.5) / raw_per_lfp
margin = int(np.ceil(gap_margin_s * fs_lfp))
for pos in gap_lfp_positions:
    lo = max(0, int(np.floor(pos)) - margin)
    hi = min(n_lfp, int(np.ceil(pos)) + margin + 1)
    lfp_valid[lo:hi] = False
spike_near_gap = ~lfp_valid[spike_lfp_index] if spike_samples.size else np.zeros(0, dtype=bool)
if forward_gaps.size:
    print(
        f"[warning] {forward_gaps.size} forward gap(s) in the samples: masked ±{gap_margin_s:g} s around each "
        f"({100 * (1 - lfp_valid.mean()):.2f}% of LFP samples, {int(spike_near_gap.sum())} spikes). "
        "Use lfp_valid.npy / spike_near_gap.npy to drop them before phase analysis."
    )

# ============================================================
# SAVE
# ============================================================

align_dir = nam.preprocessed_folder / "alignment"
align_dir.mkdir(parents=True, exist_ok=True)

np.save(align_dir / "lfp_timestamps.npy", lfp_timestamps)
np.save(align_dir / "spike_times_oe_s.npy", spike_times_oe_s)
np.save(align_dir / "spike_lfp_index.npy", spike_lfp_index)
np.save(align_dir / "lfp_valid.npy", lfp_valid)
np.save(align_dir / "spike_near_gap.npy", spike_near_gap)

crop_timing_path = Path(config["data_folder"]) / "custom_analytics" / "recording_crop_timing.npz"
crop_info = {}
if crop_timing_path.exists():
    ct = np.load(crop_timing_path)
    crop_info = {
        "remove_first_minutes": float(ct["remove_first_minutes"]),
        "probe_crop_start_frame": int(ct["probe_crop_start_frame"]),
        "probe_crop_start_s": float(ct["probe_crop_start_s"]),
    }

alignment_info = {
    "fs_raw_hz": fs_raw,
    "fs_lfp_hz": fs_lfp,
    "raw_samples_per_lfp_sample": raw_per_lfp,
    "n_raw_frames": n_raw,
    "n_lfp_frames": n_lfp,
    "n_spikes": int(len(spike_samples)),
    "spike_recording": str(spike_rec_path),
    "lfp_recording": str(lfp_rec_path),
    "kilosort_spike_times": str(ks_spike_times_path),
    "timestamps_trimmed": str(timestamps_path),
    "timestamps_source": ts_stamp["s3_provenance"]["timestamps_source"],
    "sample_numbers_source": ts_stamp["s3_provenance"]["sample_numbers_source"],
    "n_lfp_samples_past_last_raw_frame": int(n_past_end),
    "timestamp_sync_corrections": int(ts_clock_report["n_backward_corrections"]),
    "largest_timestamp_sync_correction_s": float(ts_clock_report["largest_backward_correction_s"]),
    "clock": "Open Ephys timestamps.npy (seconds), trimmed to the crop",
    "timestamps_were_sample_counts": bool(timestamps_were_counts),
    "lfp_sample_k_raw_position": "k * raw_samples_per_lfp_sample",
    "spike_lfp_index": "round(kilosort_sample / raw_samples_per_lfp_sample)",
    "n_forward_gaps": int(forward_gaps.size),
    "gap_mask_margin_s": gap_margin_s,
    "lfp_valid": "False within gap_mask_margin_s of a forward gap (filter ringing / lost data)",
    **crop_info,
}
with open(align_dir / "alignment_info.json", "w") as f:
    json.dump(alignment_info, f, indent=2)

print("\nSaved to:", align_dir)
print(f"LFP time range:   {np.nanmin(lfp_timestamps):.4f} – {np.nanmax(lfp_timestamps):.4f} s")
if spike_samples.size:
    print(f"Spike time range: {spike_times_oe_s.min():.4f} – {spike_times_oe_s.max():.4f} s")
    max_err_ms = 1000 * np.max(np.abs(spike_lfp_index * raw_per_lfp - spike_samples)) / fs_raw
    print(f"Max spike→nearest-LFP-sample offset: {max_err_ms:.3f} ms (half an LFP sample = {500 / fs_lfp:.3f} ms)")

In [ ]:
import sys
from pathlib import Path

module_dir = Path.cwd()

print("Module directory exists:", module_dir.exists())
print("Module file exists:", (module_dir / "Neuropixel_analysis_module.py").exists())

sys.path.insert(0, str(module_dir))

import Neuropixel_analysis_module as nam
print("Loaded from:", nam.__file__)